# 🍳 Fridge Chef

## Introduction

Imagine opening your fridge and asking:

> *“What can I cook tonight?”*

A useful assistant should be able to look at what you already have, understand what kind of meal
you want, find a suitable recipe, identify any missing ingredients, and prepare a shopping list.

That sounds simple, but there is an important difference between requests such as:

> *“Cook something with eggs.”*

and:

> *“I want something comforting for a cold winter evening.”*

The first can be handled with a few explicit rules. The second requires interpreting what the user
means.

In this notebook, **Fridge Chef** will be our small environment for exploring that difference.

## From a request to a sequence of actions

Fridge Chef does not solve the task in a single step. It can perform a small set of actions:
inspect the fridge, explore recipes, check ingredients, update a shopping list, and finish the task.

The interesting part is deciding:

> **Given what I know right now, what should I do next?**

An agent repeatedly makes that decision, performs an action, observes the result, and decides again.

**request → choose an action → observe the result → choose the next action → … → done**

Later, we'll implement this decision-making in different ways and compare how they behave.

In [1]:
#@title 🍳 Fridge Chef: the loop, in one picture { display-mode: "form" }
from IPython.display import display, HTML

_HERO_CSS = """
<style>
.fc-hero{--bg:#fff;--sunk:#f6f7f9;--fg:#1f2328;--dim:#5d6672;--line:#dfe3e8;
  --accent:#1a56db;--bar:#1a56db;--t1:#177245;--t2:#6d3fd4;--t3:#1565c0;--t4:#a85a00;--t5:#b3261e;--t6:#0f766e;
  background:var(--bg);color:var(--fg);border:1px solid var(--line);border-radius:14px;
  padding:22px 24px;max-width:820px;
  font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,Helvetica,Arial,sans-serif;
  line-height:1.55;box-sizing:border-box}
@media (prefers-color-scheme:dark){.fc-hero{--bg:#161a1f;--sunk:#1e242b;--fg:#e6edf3;
  --dim:#9aa5b1;--line:#2d353e;--accent:#6ea8ff;--bar:#2547a8;--t1:#5fd08a;--t2:#b49bff;--t3:#79b8ff;
  --t4:#e0a458;--t5:#ff8a80;--t6:#5eead4}}
.fc-hero h2{font-size:16px;margin:22px 0 8px;color:var(--fg)}
.fc-hero h2.top{font-size:19px;margin:0 0 6px;color:var(--accent);letter-spacing:-.2px}
.fc-hero p,.fc-hero li{font-size:14px;color:var(--dim);margin:8px 0}
.fc-hero strong{color:var(--fg)}
.fc-hero .lede{font-size:15px}
.fc-hero .panel{background:var(--sunk);border:1px solid var(--line);border-radius:10px;
  padding:14px 16px;margin:14px 0}
.fc-hero svg{display:block;width:100%;height:auto;max-width:740px;margin:6px auto}
.fc-hero ol{padding-left:20px;margin:8px 0}
.fc-hero code{background:var(--sunk);border:1px solid var(--line);border-radius:5px;
  padding:1px 5px;font-size:12.5px;color:var(--fg)}
</style>
"""

_HERO_SVG = """
<svg viewBox="0 0 760 330" role="img"
     aria-label="Diagram: a user request reaches a decision maker, which chooses one of six actions; the result of that action travels back to the decision maker, which decides again.">
  <defs>
    <marker id="fcArrow" markerWidth="9" markerHeight="9" refX="7" refY="3"
            orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="currentColor"/></marker>
  </defs>
  <g font-family="-apple-system,BlinkMacSystemFont,Segoe UI,Roboto,sans-serif">

    <g color="var(--dim)">
      <rect x="230" y="6" width="300" height="38" rx="19"
            fill="var(--sunk)" stroke="var(--line)"/>
      <text x="380" y="30" text-anchor="middle" font-size="14" fill="var(--fg)">
        💬  "What can I cook tonight?"</text>
      <path d="M380,46 L380,68" stroke="currentColor" stroke-width="1.6"
            fill="none" marker-end="url(#fcArrow)"/>
    </g>

    <rect x="230" y="70" width="300" height="58" rx="12"
          fill="var(--bar)" stroke="var(--bar)"/>
    <text x="380" y="94" text-anchor="middle" font-size="15" font-weight="700"
          fill="#fff">Decision maker</text>
    <text x="380" y="113" text-anchor="middle" font-size="11" fill="#fff"
          opacity=".88">reads the current state, chooses the next action</text>

    <g color="var(--dim)" stroke="currentColor" stroke-width="1.3" fill="none">
      <path d="M380,130 C380,152 85,150 85,176" marker-end="url(#fcArrow)"/>
      <path d="M380,130 C380,152 203,150 203,176" marker-end="url(#fcArrow)"/>
      <path d="M380,130 C380,152 321,150 321,176" marker-end="url(#fcArrow)"/>
      <path d="M380,130 C380,152 439,150 439,176" marker-end="url(#fcArrow)"/>
      <path d="M380,130 C380,152 557,150 557,176" marker-end="url(#fcArrow)"/>
      <path d="M380,130 C380,152 675,150 675,176" marker-end="url(#fcArrow)"/>
    </g>

    <g>
      <g><rect x="30" y="180" width="110" height="62" rx="10" fill="var(--sunk)"
               stroke="var(--t1)"/>
         <text x="85" y="201" text-anchor="middle" font-size="16">🥚</text>
         <text x="85" y="219" text-anchor="middle" font-size="10" font-weight="700"
               fill="var(--t1)">look in the fridge</text>
         <text x="85" y="233" text-anchor="middle" font-size="8.5"
               fill="var(--dim)" font-family="ui-monospace,SFMono-Regular,Menlo,monospace">check_fridge</text></g>
      <g><rect x="148" y="180" width="110" height="62" rx="10" fill="var(--sunk)"
               stroke="var(--t6)"/>
         <text x="203" y="201" text-anchor="middle" font-size="16">📚</text>
         <text x="203" y="219" text-anchor="middle" font-size="10" font-weight="700"
               fill="var(--t6)">see all recipes</text>
         <text x="203" y="233" text-anchor="middle" font-size="8.5"
               fill="var(--dim)" font-family="ui-monospace,SFMono-Regular,Menlo,monospace">list_recipes</text></g>
      <g><rect x="266" y="180" width="110" height="62" rx="10" fill="var(--sunk)"
               stroke="var(--t2)"/>
         <text x="321" y="201" text-anchor="middle" font-size="16">🔍</text>
         <text x="321" y="219" text-anchor="middle" font-size="10" font-weight="700"
               fill="var(--t2)">search recipes</text>
         <text x="321" y="233" text-anchor="middle" font-size="8.5"
               fill="var(--dim)" font-family="ui-monospace,SFMono-Regular,Menlo,monospace">search_recipes</text></g>
      <g><rect x="384" y="180" width="110" height="62" rx="10" fill="var(--sunk)"
               stroke="var(--t3)"/>
         <text x="439" y="201" text-anchor="middle" font-size="16">📖</text>
         <text x="439" y="219" text-anchor="middle" font-size="10" font-weight="700"
               fill="var(--t3)">check ingredients</text>
         <text x="439" y="233" text-anchor="middle" font-size="8.5"
               fill="var(--dim)" font-family="ui-monospace,SFMono-Regular,Menlo,monospace">get_ingredients</text></g>
      <g><rect x="502" y="180" width="110" height="62" rx="10" fill="var(--sunk)"
               stroke="var(--t4)"/>
         <text x="557" y="201" text-anchor="middle" font-size="16">🛒</text>
         <text x="557" y="219" text-anchor="middle" font-size="10" font-weight="700"
               fill="var(--t4)">update the list</text>
         <text x="557" y="233" text-anchor="middle" font-size="8.5"
               fill="var(--dim)" font-family="ui-monospace,SFMono-Regular,Menlo,monospace">add_to_list</text></g>
      <g><rect x="620" y="180" width="110" height="62" rx="10" fill="var(--sunk)"
               stroke="var(--t5)"/>
         <text x="675" y="201" text-anchor="middle" font-size="16">✅</text>
         <text x="675" y="219" text-anchor="middle" font-size="10" font-weight="700"
               fill="var(--t5)">finish the task</text>
         <text x="675" y="233" text-anchor="middle" font-size="8.5"
               fill="var(--dim)" font-family="ui-monospace,SFMono-Regular,Menlo,monospace">done</text></g>
    </g>

    <!-- results collect below the actions, then return up the outside to the decision maker -->
    <g color="var(--accent)" fill="none">
      <path d="M85,244 C85,288 380,298 380,298 C380,298 675,288 675,244"
            stroke="currentColor" stroke-width="1.5" stroke-dasharray="5 4"/>
      <path d="M292,298 L38,298 Q14,298 14,274 L14,128 Q14,104 38,104 L218,104"
            stroke="currentColor" stroke-width="1.8" marker-end="url(#fcArrow)"/>
      <rect x="292" y="285" width="176" height="26" rx="13"
            fill="var(--bg)" stroke="currentColor"/>
      <text x="380" y="302" text-anchor="middle" font-size="11.5" font-weight="700"
            fill="var(--accent)">the result goes back in ↻</text>
    </g>
  </g>
</svg>
"""

display(HTML(_HERO_CSS + """
<div class="fc-hero">
  <h2 class="top">The agent loop</h2>
  <p class="lede">One request, many small steps. The <strong>decision maker</strong> looks at what
  is known so far, picks <em>one</em> action, and reads the result, then decides again, until
  the task is finished.</p>
""" + _HERO_SVG + """
  <div class="panel">
    <p style="margin:0">Every version of Fridge Chef you build uses <strong>this same loop</strong>
    and these same six actions. The only thing that changes is
    <strong>who plays the decision maker</strong>.</p>
  </div>

  <h2>Your route</h2>
  <ol>
    <li><strong>Rules decide.</strong> Hand-written <code>if/else</code>. Fine on
    <em>"cook something with eggs"</em>; it falls apart the moment a request needs interpreting.</li>
    <li><strong>An LLM decides.</strong> Same six actions, same loop, a model in the middle.
    Suddenly <em>"comforting, cold evening"</em> works.</li>
    <li><strong>Swap the model.</strong> Point the same loop at a model running on this
    machine's GPU instead of a cloud API, and see what you gain and lose.</li>
    <li><strong>Evaluate it.</strong> Because one good demo proves nothing.</li>
  </ol>
</div>
"""))

In [2]:
#@title Setup 1/3 · The kitchen simulation (run me) { display-mode: "form" }
# ---------------------------------------------------------------------------
# Everything the exercise needs is right here: no repo to clone, no downloads.
# You never have to edit this cell, but you're very welcome to read it.
# ---------------------------------------------------------------------------
import importlib.util, json, re, subprocess, sys, time
from dataclasses import dataclass, field

if importlib.util.find_spec("openai") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai"], check=True)

# ── The scenario ───────────────────────────────────────────────────────────
# What's actually in the fridge. The same twelve items for everyone, every run,
# so your results are reproducible and comparable with the person next to you.
FRIDGE_CONTENTS = [
    "eggs", "milk", "butter", "cheddar cheese", "bread", "tomatoes",
    "onions", "garlic", "bell pepper", "chicken breast", "rice", "soy sauce",
]

# recipe -> everything it needs
INGREDIENTS_DB = {
    "omelette":             ["eggs", "butter", "cheddar cheese", "bell pepper", "onions", "salt", "pepper"],
    "fried rice":           ["rice", "eggs", "soy sauce", "onions", "garlic", "sesame oil", "green onions"],
    "french toast":         ["bread", "eggs", "milk", "butter", "cinnamon", "sugar", "vanilla extract"],
    "egg salad sandwich":   ["eggs", "bread", "mayonnaise", "mustard", "salt", "pepper"],
    "shakshuka":            ["eggs", "tomatoes", "onions", "garlic", "bell pepper", "cumin", "paprika", "olive oil"],
    "carbonara":            ["pasta", "eggs", "parmesan cheese", "pancetta", "black pepper", "garlic"],
    "pancakes":             ["flour", "milk", "eggs", "butter", "sugar", "baking powder", "salt"],
    "grilled cheese":       ["bread", "butter", "cheddar cheese"],
    "garlic bread":         ["bread", "butter", "garlic", "parsley"],
    "mac and cheese":       ["pasta", "milk", "cheddar cheese", "butter", "flour", "salt"],
    "bruschetta":           ["bread", "tomatoes", "garlic", "olive oil", "basil", "salt"],
    "tomato soup":          ["tomatoes", "onions", "garlic", "butter", "cream", "basil", "salt", "pepper"],
    "stir fry":             ["bell pepper", "onions", "garlic", "soy sauce", "sesame oil", "cornstarch"],
    "chicken stir fry":     ["chicken breast", "bell pepper", "soy sauce", "garlic", "onions", "sesame oil", "cornstarch", "ginger"],
    "chicken fried rice":   ["chicken breast", "rice", "eggs", "soy sauce", "onions", "garlic", "sesame oil"],
    "chicken quesadilla":   ["chicken breast", "cheddar cheese", "tortilla", "bell pepper", "onions", "salsa"],
    "grilled chicken salad":["chicken breast", "lettuce", "tomatoes", "cucumber", "olive oil", "lemon juice"],
    "chicken curry":        ["chicken breast", "rice", "onions", "garlic", "coconut milk", "curry powder", "ginger"],
    "rice bowl":            ["rice", "soy sauce", "sesame oil", "green onions", "eggs"],
    "stuffed peppers":      ["bell pepper", "rice", "tomatoes", "onions", "cheddar cheese", "ground beef"],
    "fajitas":              ["bell pepper", "onions", "chicken breast", "tortilla", "lime", "cumin", "chili powder"],
    "pasta alfredo":        ["pasta", "butter", "garlic", "parmesan cheese", "cream", "salt", "pepper"],
    "quesadilla":           ["tortilla", "cheddar cheese", "salsa"],
    "caprese salad":        ["tomatoes", "mozzarella", "basil", "olive oil", "balsamic vinegar"],
    "salsa":                ["tomatoes", "onions", "garlic", "jalapeno", "cilantro", "lime juice"],
    "french onion soup":    ["onions", "butter", "beef broth", "bread", "gruyere cheese", "thyme"],
    "teriyaki chicken":     ["chicken breast", "soy sauce", "sugar", "garlic", "ginger", "cornstarch"],
    "spaghetti bolognese":  ["pasta", "ground beef", "tomatoes", "onions", "garlic", "olive oil", "basil"],
}

# ingredient -> every recipe that uses it (built by inverting the table above)
RECIPE_DB: dict[str, list[str]] = {}
for _recipe, _ingredients in INGREDIENTS_DB.items():
    for _ing in _ingredients:
        RECIPE_DB.setdefault(_ing, []).append(_recipe)


def _resolve(query: str, candidates) -> str | None:
    """Map a loose name onto a known key: 'chicken' -> 'chicken breast', 'egg' -> 'eggs'.

    Real tools are forgiving about their inputs, because models are not careful
    about spelling. Returns the canonical key, or None if nothing is close.
    """
    q = " ".join(query.lower().split())
    if not q:
        return None
    lookup = {c.lower(): c for c in candidates}
    if q in lookup:                                    # exact
        return lookup[q]
    q_words = set(q.split())
    hits = [(len(set(low.split()) & q_words), -len(low), orig)
            for low, orig in lookup.items() if set(low.split()) & q_words]
    if hits:                                           # shares a whole word
        return max(hits)[2]
    for low, orig in lookup.items():                   # substring either way
        if q in low or low in q:
            return orig
    return None


# ── State ──────────────────────────────────────────────────────────────────
@dataclass
class ChefState:
    """Everything the agent has learned so far. Starts empty; the tools fill it in."""
    fridge_contents:   list[str] = field(default_factory=list)
    possible_recipes:  list[str] = field(default_factory=list)
    chosen_recipe:     str = ""
    needed_ingredients:list[str] = field(default_factory=list)
    shopping_list:     list[str] = field(default_factory=list)
    unavailable:       list[str] = field(default_factory=list)   # asked for, uncookable
    shopping_done:     bool = False

    @property
    def is_complete(self) -> bool:
        return self.shopping_done


class KitchenWorld:
    """The kitchen itself. The agent can only learn about it by calling tools."""

    def __init__(self, shuffle_seed: int | None = None):
        if shuffle_seed is None:
            self.fridge = list(FRIDGE_CONTENTS)        # the default: same for everyone
        else:
            import random                              # opt-in chaos, still reproducible
            pool = sorted({i for v in INGREDIENTS_DB.values() for i in v})
            self.fridge = random.Random(shuffle_seed).sample(pool, len(FRIDGE_CONTENTS))

    def get_fridge_contents(self) -> list[str]:
        return list(self.fridge)

    def all_recipes(self) -> list[str]:
        """Every recipe name in the cookbook, so an agent can browse instead of guess."""
        return list(INGREDIENTS_DB)

    def search_recipes(self, ingredient: str) -> tuple[str | None, list[str]]:
        """-> (the ingredient name we actually matched, recipes using it)."""
        key = _resolve(ingredient, RECIPE_DB)
        return (key, list(RECIPE_DB[key])) if key else (None, [])

    def get_recipe_ingredients(self, recipe: str) -> tuple[str | None, list[str]]:
        """-> (the recipe name we actually matched, its full ingredient list)."""
        key = _resolve(recipe, INGREDIENTS_DB)
        return (key, list(INGREDIENTS_DB[key])) if key else (None, [])


# ── Tool result messages ───────────────────────────────────────────────────
# One place, used by BOTH the rule-based agent and your LLM tools, so the same
# action always reads the same way. This text is the agent's only view of the
# world, so writing it well is part of building a good tool.
def fmt_fridge(items: list[str]) -> str:
    return f"Fridge ({len(items)} items): {', '.join(items)}"


def fmt_recipes(recipes: list[str]) -> str:
    return f"The cookbook has {len(recipes)} recipes: {', '.join(recipes)}"


def fmt_search(query: str, matched: str | None, recipes: list[str], fridge: list[str]) -> str:
    if not recipes:
        return (f"No recipes found for '{query}'. "
                f"Search for something you actually have: {', '.join(fridge)}.")
    note = f" (matched '{query}' -> '{matched}')" if matched.lower() != query.lower().strip() else ""
    return f"Found {len(recipes)} recipes using '{matched}'{note}: {', '.join(recipes)}"


def fmt_ingredients(recipe: str, matched: str | None,
                    ingredients: list[str], fridge: list[str]) -> str:
    if not ingredients:
        return (f"Recipe '{recipe}' not found. Call search_recipes first, "
                f"then pick one of the names it gives you.")
    have = [i for i in ingredients if i.lower() in {f.lower() for f in fridge}]
    missing = [i for i in ingredients if i not in have]
    lines = [f"Ingredients for '{matched}': {', '.join(ingredients)}",
             f"Already in fridge: {', '.join(have) if have else '(none)'}"]
    lines.append(f"Missing (add these to the shopping list): {', '.join(missing)}"
                 if missing else "Missing: nothing. You have everything!")
    return "\n".join(lines)


def fmt_added(item: str, shopping_list: list[str]) -> str:
    return f"Added '{item}' to shopping list. Current list: {', '.join(shopping_list)}"


def fmt_done(recipe: str, shopping_list: list[str], unavailable=()) -> str:
    lines = ["Workflow complete!", f"Recipe: {recipe}"]
    lines.append(f"Shopping list: {', '.join(shopping_list)}" if shopping_list
                 else "Shopping list: empty. Everything is already in the fridge!")
    if unavailable:
        lines.append(f"Could not be cooked, so buy it instead: {', '.join(unavailable)}")
    return "\n".join(lines)


# ── The six tools ──────────────────────────────────────────────────────────
@dataclass
class ToolResult:
    success: bool
    message: str


class KitchenTools:
    """The five actions an agent can take. Each one reads/writes `chef` and
    returns a plain string, and that string is what the agent sees next."""

    def __init__(self, chef: ChefState, world: KitchenWorld):
        self.chef, self.world = chef, world

    def check_fridge(self) -> ToolResult:
        self.chef.fridge_contents = self.world.get_fridge_contents()
        return ToolResult(True, fmt_fridge(self.chef.fridge_contents))

    def list_recipes(self) -> ToolResult:
        names = self.world.all_recipes()
        self.chef.possible_recipes = names
        return ToolResult(True, fmt_recipes(names))

    def search_recipes(self, ingredient: str = "") -> ToolResult:
        if not ingredient.strip():
            return ToolResult(False, "Please say which ingredient to search for.")
        matched, recipes = self.world.search_recipes(ingredient)
        if recipes:
            self.chef.possible_recipes = recipes
        return ToolResult(bool(recipes),
                          fmt_search(ingredient, matched, recipes, self.world.get_fridge_contents()))

    def get_ingredients(self, recipe: str = "") -> ToolResult:
        if not recipe.strip():
            return ToolResult(False, "Please say which recipe to look up.")
        matched, ingredients = self.world.get_recipe_ingredients(recipe)
        if ingredients:
            self.chef.chosen_recipe = matched
            self.chef.needed_ingredients = ingredients
        return ToolResult(bool(ingredients),
                          fmt_ingredients(recipe, matched, ingredients, self.chef.fridge_contents))

    def add_to_shopping_list(self, item: str = "") -> ToolResult:
        item = item.strip()
        if not item:
            return ToolResult(False, "Please say which item to add.")
        if item.lower() in {f.lower() for f in self.chef.fridge_contents}:
            return ToolResult(False, f"'{item}' is already in your fridge, no need to buy it.")
        if item.lower() in {s.lower() for s in self.chef.shopping_list}:
            return ToolResult(False, f"'{item}' is already on your shopping list.")
        self.chef.shopping_list.append(item)
        return ToolResult(True, fmt_added(item, self.chef.shopping_list))

    def done(self, unavailable: str = "") -> ToolResult:
        """`unavailable`: anything the user asked for that the kitchen cannot cook.

        Naming it is all the agent has to do, and we put it on the shopping list
        ourselves. Asking a model to remember an extra tool call five steps later
        is a hope; making it an argument of the call it must make anyway is a
        guarantee."""
        if not self.chef.chosen_recipe:
            return ToolResult(False, "No recipe chosen yet. Call get_ingredients(recipe=...) first.")
        extras = [i.strip() for i in unavailable.split(",") if i.strip()]
        on_list = {s.lower() for s in self.chef.shopping_list}
        added = [i for i in extras if i.lower() not in on_list]
        self.chef.shopping_list.extend(added)
        self.chef.unavailable = extras
        self.chef.shopping_done = True
        return ToolResult(True, fmt_done(self.chef.chosen_recipe,
                                         self.chef.shopping_list, extras))

    def execute(self, tool_name: str, args: dict) -> ToolResult:
        """Dispatch by name. This is what an agent loop actually calls."""
        table = {
            "check_fridge":         lambda: self.check_fridge(),
            "list_recipes":         lambda: self.list_recipes(),
            "search_recipes":       lambda: self.search_recipes(args.get("ingredient", "")),
            "get_ingredients":      lambda: self.get_ingredients(args.get("recipe", "")),
            "add_to_shopping_list": lambda: self.add_to_shopping_list(args.get("item", "")),
            "done":                 lambda: self.done(args.get("unavailable", "")),
        }
        fn = table.get(str(tool_name).lower().strip())
        if fn is None:
            return ToolResult(False, f"No such tool '{tool_name}'. Available: {', '.join(table)}.")
        return fn()


TOOL_NAMES = ["check_fridge", "list_recipes", "search_recipes", "get_ingredients",
              "add_to_shopping_list", "done"]


def parse_action(text: str) -> tuple[str, dict]:
    """Read 'ACTION: search_recipes(ingredient="eggs")' into ('search_recipes', {...}).

    Phase 1 only. In Phase 2 the model hands us structured JSON and this
    string-scraping disappears entirely, which is rather the point.
    """
    m = re.search(r'ACTION:\s*(\w+)\((.*?)\)', text, re.DOTALL)
    if not m:
        simple = re.search(r'ACTION:\s*(\w+)', text)
        if simple:
            return simple.group(1), {}
        raise ValueError(
            f"Could not find an 'ACTION: tool_name(...)' call in this text:\n{text!r}\n"
            f"Your think function must return a string like: ACTION: check_fridge()")
    args = {k: v for k, v in re.findall(r'(\w+)\s*=\s*["\']([^"\']*)["\']', m.group(2))}
    return m.group(1), args


def create_game(shuffle_seed: int | None = None):
    """Fresh chef, fresh kitchen, fresh tools."""
    world = KitchenWorld(shuffle_seed)
    chef = ChefState()
    return chef, world, KitchenTools(chef, world)


print(f"Kitchen loaded · {len(FRIDGE_CONTENTS)} items in the fridge · "
      f"{len(INGREDIENTS_DB)} recipes · {len(TOOL_NAMES)} tools")

Kitchen loaded · 12 items in the fridge · 28 recipes · 6 tools


In [3]:
#@title Setup 2/3 · The live dashboard (run me) { display-mode: "form" }
# ---------------------------------------------------------------------------
# Draws the agent's state as it works. Nothing here affects agent behaviour;
# it just makes the loop visible. One stylesheet, opaque panels, so it stays
# readable in Colab's light theme, Colab's dark theme, and plain Jupyter.
# ---------------------------------------------------------------------------
import html as _html
from IPython.display import display, update_display, clear_output, HTML

FC_CSS = """
<style>
.fc{--bg:#fff;--sunk:#f6f7f9;--fg:#1f2328;--dim:#5d6672;--faint:#6e7781;
    --line:#dfe3e8;--accent:#1a56db;--ok:#177245;--warn:#a85a00;--bad:#b3261e;
    --t-check:#177245;--t-list:#0f766e;--t-search:#6d3fd4;--t-ingr:#1565c0;--t-shop:#a85a00;--t-done:#b3261e;
    --bar:#1a56db;--ok-solid:#177245;--warn-solid:#a85a00;--bad-solid:#b3261e;
    --chip-ok-bg:#e8f5ec;--chip-ok-fg:#14532d;--chip-warn-bg:#fdf1e0;--chip-warn-fg:#7c4300;
    --chip-bad-bg:#fdecea;--chip-bad-fg:#8c1d18;
    background:var(--bg);color:var(--fg);border:1px solid var(--line);border-radius:14px;
    max-width:860px;overflow:hidden;box-sizing:border-box;
    font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,Helvetica,Arial,sans-serif;
    font-size:14px;line-height:1.5}
@media (prefers-color-scheme:dark){.fc{--bg:#161a1f;--sunk:#1e242b;--fg:#e6edf3;--dim:#9aa5b1;
    --faint:#7d8896;--line:#2d353e;--accent:#6ea8ff;--ok:#5fd08a;--warn:#e0a458;--bad:#ff8a80;
    --chip-ok-bg:#12301f;--chip-ok-fg:#7ee2a8;--chip-warn-bg:#33240e;--chip-warn-fg:#f0c07a;
    --chip-bad-bg:#3a1512;--chip-bad-fg:#ffab9e;
    --t-check:#5fd08a;--t-list:#5eead4;--t-search:#b49bff;--t-ingr:#79b8ff;--t-shop:#e0a458;--t-done:#ff8a80;
    --bar:#2547a8;--ok-solid:#1a7a44;--warn-solid:#9a5b00;--bad-solid:#b3261e}}
.fc *{box-sizing:border-box}
.fc code,.fc pre{font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace}

.fc-bar{background:var(--bar);color:#fff;padding:10px 16px;display:flex;
        flex-wrap:wrap;gap:8px;align-items:center;justify-content:space-between}
.fc-bar b{font-size:15px;letter-spacing:.2px}
.fc-tag{background:rgba(255,255,255,.22);border-radius:20px;padding:3px 11px;font-size:11.5px}
.fc-ask{background:rgba(255,255,255,.22);border-radius:20px;padding:3px 12px;font-size:12px;
        max-width:100%;overflow-wrap:anywhere}

.fc-banner{padding:7px 16px;text-align:center;font-weight:700;font-size:13px;color:#fff}
.fc-banner.ok{background:var(--ok-solid)} .fc-banner.warn{background:var(--warn-solid)}
.fc-banner.bad{background:var(--bad-solid)}

.fc-pipe{display:flex;flex-wrap:wrap;gap:6px;padding:11px 16px;background:var(--sunk);
         border-bottom:1px solid var(--line)}
.fc-step{display:flex;align-items:center;gap:5px;padding:3px 10px;border-radius:20px;
         font-size:11.5px;border:1px solid var(--line);color:var(--faint);background:var(--bg)}
.fc-step.on{border-color:currentColor;color:var(--ok);font-weight:600}

.fc-body{display:flex;flex-wrap:wrap;align-items:stretch}
.fc-left{flex:1 1 250px;min-width:230px;padding:14px 16px;border-right:1px solid var(--line)}
.fc-right{flex:2 1 340px;min-width:280px;padding:14px 16px;display:flex;
          flex-direction:column;gap:11px}
@media (max-width:560px){.fc-left{border-right:none;border-bottom:1px solid var(--line)}}

.fc-h{font-size:10.5px;font-weight:700;letter-spacing:.9px;text-transform:uppercase;
      color:var(--faint);margin-bottom:7px}
.fc-empty{color:var(--faint);font-size:12px;font-style:italic}
.fc-chip{display:inline-block;border-radius:14px;padding:2px 9px;margin:2px 2px 2px 0;
         font-size:11.5px;border:1px solid transparent}
.fc-chip.n{background:var(--sunk);border-color:var(--line);color:var(--fg)}
.fc-chip.g{background:var(--chip-ok-bg);color:var(--chip-ok-fg)}
.fc-chip.a{background:var(--chip-warn-bg);color:var(--chip-warn-fg)}
.fc-chip.r{background:var(--chip-bad-bg);color:var(--chip-bad-fg)}
.fc-chip.x{background:var(--chip-warn-bg);color:var(--chip-warn-fg);
           border:1px dashed currentColor}
.fc-note{font-size:10.5px;color:var(--faint);margin-top:6px;line-height:1.4}

.fc-meter{display:flex;align-items:center;gap:9px;font-size:11.5px;color:var(--faint)}
.fc-track{flex:1;height:7px;border-radius:6px;background:var(--sunk);
          border:1px solid var(--line);overflow:hidden}
.fc-fill{height:100%;background:var(--accent);min-width:2px}

.fc-panel{background:var(--sunk);border:1px solid var(--line);border-radius:10px;padding:10px 13px}
.fc-recipe{font-size:17px;font-weight:700;color:var(--accent);margin-bottom:2px}
.fc-act{border:1px solid var(--line);border-left:4px solid var(--tc,var(--accent));
        border-radius:9px;padding:10px 13px;background:var(--bg)}
.fc-act code{font-size:13px;font-weight:700;color:var(--tc,var(--accent));overflow-wrap:anywhere}
.fc-out{margin-top:7px;padding:7px 10px;border-radius:7px;background:var(--sunk);
        border-left:3px solid var(--line);font-size:12px;color:var(--dim);white-space:pre-wrap}
.fc-out.ok{border-left-color:var(--ok)} .fc-out.bad{border-left-color:var(--bad)}

.fc-log{border-top:1px solid var(--line);padding:10px 16px;background:var(--sunk)}
.fc-log details{border-bottom:1px solid var(--line);padding:5px 0}
.fc-log details:last-child{border-bottom:none}
.fc-log summary{cursor:pointer;list-style:none;display:flex;align-items:center;gap:7px;
                font-size:12px}
.fc-log summary::-webkit-details-marker{display:none}
.fc-log .num{background:var(--tc,var(--accent));color:var(--bg);border-radius:10px;padding:0 7px;
             font-size:10px;font-weight:700;min-width:20px;text-align:center}
.fc-log .peek{color:var(--faint);font-size:11.5px;overflow:hidden;text-overflow:ellipsis;
              white-space:nowrap;flex:1}
.fc-log pre{margin:6px 0 2px 34px;font-size:11.5px;color:var(--dim);white-space:pre-wrap}

.fc-err{margin:14px 16px;border:1px solid var(--bad);border-left:4px solid var(--bad);
        border-radius:9px;padding:11px 14px;background:var(--chip-bad-bg)}
.fc-err b{color:var(--chip-bad-fg);font-size:13px}
.fc-err pre{margin:7px 0 0;font-size:11.5px;color:var(--chip-bad-fg);white-space:pre-wrap;
            overflow-x:auto}

.fc-tbl{width:100%;border-collapse:collapse;font-size:13px}
.fc-tbl th,.fc-tbl td{padding:8px 12px;text-align:left;border-bottom:1px solid var(--line)}
.fc-tbl th{font-size:10.5px;letter-spacing:.9px;text-transform:uppercase;color:var(--faint);
           background:var(--sunk)}
.fc-tbl td.num{text-align:right;font-variant-numeric:tabular-nums}
.fc pre.code{background:var(--sunk);border:1px solid var(--line);border-radius:8px;
  padding:11px 13px;margin:8px 18px;font-size:11.5px;color:var(--dim);
  white-space:pre;overflow-x:auto;line-height:1.45}
.fc-pad{padding:16px 18px}
.fc-pad h3{margin:0 0 8px;font-size:16px;color:var(--accent)}
.fc-pad p{margin:8px 0;color:var(--dim)}
.fc-pad strong{color:var(--fg)}
</style>
"""

# colours are CSS variables, not hex, so every tool re-colours itself in dark mode
TOOL_STYLE = {
    "check_fridge":         ("🥚", "var(--t-check)",  "Check Fridge"),
    "list_recipes":         ("📚", "var(--t-list)",   "List Recipes"),
    "search_recipes":       ("🔍", "var(--t-search)", "Search Recipes"),
    "get_ingredients":      ("📖", "var(--t-ingr)",   "Get Ingredients"),
    "add_to_shopping_list": ("🛒", "var(--t-shop)",   "Shopping List"),
    "done":                 ("✅", "var(--t-done)",   "Done"),
}
_FOOD = {"eggs": "🥚", "milk": "🥛", "butter": "🧈", "cheddar cheese": "🧀", "bread": "🍞",
         "tomatoes": "🍅", "onions": "🧅", "garlic": "🧄", "bell pepper": "🫑",
         "chicken breast": "🍗", "rice": "🍚", "soy sauce": "🍶"}


def _esc(s) -> str:
    return _html.escape(str(s))


def _tool_of(action: str) -> tuple[str, str, str]:
    for name, style in TOOL_STYLE.items():
        if action.startswith(name):
            return style
    return ("⚙️", "var(--dim)", "Tool")


def fc_wrap(body: str) -> str:
    """Wrap arbitrary markup in the notebook's stylesheet + theme container."""
    return FC_CSS + f'<div class="fc">{body}</div>'


def _chips(items, kind="n", emoji=False) -> str:
    if not items:
        return '<span class="fc-empty">empty</span>'
    return "".join(
        f'<span class="fc-chip {kind}">'
        f'{_FOOD.get(str(i).lower(), "") + " " if emoji else ""}{_esc(i)}</span>'
        for i in items)


def build_dashboard(request, chef, step, max_steps, brain="if/else rules",
                    action=None, result=None, success=None,
                    log=None, status=None, error=None) -> str:
    """Render the whole agent state as one self-contained HTML block."""
    steps = [("Check fridge", bool(chef.fridge_contents)),
             ("Find recipes", bool(chef.possible_recipes)),
             ("Choose recipe", bool(chef.chosen_recipe)),
             ("Read ingredients", bool(chef.needed_ingredients)),
             ("Shop", bool(chef.shopping_list) or chef.shopping_done),
             ("Finish", chef.is_complete)]
    pipe = "".join(
        f'<span class="fc-step{" on" if done else ""}">{"✓" if done else "○"} {label}</span>'
        for label, done in steps)

    banner = ""
    if status == "complete":
        banner = '<div class="fc-banner ok">✅ COMPLETE: ready to cook</div>'
    elif status == "timeout":
        banner = (f'<div class="fc-banner warn">⏰ STOPPED after {max_steps} steps '
                  f'without calling done()</div>')
    elif status == "notools":
        banner = ('<div class="fc-banner warn">💬 The model answered with prose instead of '
                  'calling a tool, twice. Sharpen the system prompt and try again.</div>')
    elif status == "stuck":
        banner = ('<div class="fc-banner warn">🔁 STUCK: same action four times running, '
                  'nothing changed. A condition that never becomes false: check you are '
                  'testing the field that this action\'s own tool fills in.</div>')
    elif status == "error":
        banner = '<div class="fc-banner bad">💥 YOUR CODE RAISED AN ERROR: see below</div>'

    err_html = ""
    if error:
        err_html = (f'<div class="fc-err"><b>{_esc(error[0])}</b>'
                    f'<pre>{_esc(error[1])}</pre></div>')

    # recipe panel
    if chef.chosen_recipe:
        have = {i.lower() for i in chef.fridge_contents}
        cart = {i.lower() for i in chef.shopping_list}
        ings = "".join(
            f'<span class="fc-chip {"g" if i.lower() in have else "a" if i.lower() in cart else "r"}">'
            f'{"✓" if i.lower() in have else "🛒" if i.lower() in cart else "✗"} {_esc(i)}</span>'
            for i in chef.needed_ingredients)
        recipe = (f'<div class="fc-recipe">🍽️ {_esc(chef.chosen_recipe.title())}</div>'
                  f'<div>{ings}</div>')
    else:
        recipe = '<span class="fc-empty">no recipe chosen yet</span>'

    # current action
    act_html = ""
    if action:
        emoji, colour, _ = _tool_of(action)
        cls = "ok" if success else "bad" if success is False else ""
        act_html = (f'<div class="fc-act" style="--tc:{colour}">'
                    f'{emoji} <code>{_esc(action)}</code>'
                    f'<div class="fc-out {cls}">{_esc(result or "")}</div></div>')

    # history: full result behind a disclosure triangle, never truncated away
    rows = ""
    for e in (log or []):
        emoji, colour, _ = _tool_of(e["action"])
        peek = " ".join(str(e["result"]).split())
        rows += (f'<details style="--tc:{colour}"><summary>'
                 f'<span class="num">{e["step"]}</span>{emoji} '
                 f'<code>{_esc(e["action"])}</code>'
                 f'<span class="peek">{_esc(peek[:70])}</span></summary>'
                 f'<pre>{_esc(e["result"])}</pre></details>')
    if not rows:
        rows = '<span class="fc-empty">nothing yet</span>'

    # the shopping list holds two different kinds of thing: ingredients the chosen
    # recipe is missing, and things the user asked for that nothing can cook. Show
    # the difference, otherwise "ice cream" looks like an ingredient of french toast.
    uncookable = {i.lower() for i in getattr(chef, "unavailable", [])}
    if chef.shopping_list:
        shop_html = "".join(
            f'<span class="fc-chip {"x" if i.lower() in uncookable else "a"}">'
            f'{"✱ " if i.lower() in uncookable else ""}{_esc(i)}</span>'
            for i in chef.shopping_list)
        if uncookable:
            shop_html += ('<div class="fc-note">✱ asked for, but nothing in the cookbook '
                          'makes it (buy it ready-made)</div>')
    else:
        shop_html = '<span class="fc-empty">empty</span>'

    pct = min(100, int(step / max(max_steps, 1) * 100))
    return fc_wrap(f"""
  <div class="fc-bar">
    <b>🧑‍🍳 Fridge Chef</b>
    <span class="fc-tag">brain: {_esc(brain)}</span>
    <span class="fc-ask">💬 {_esc(request)}</span>
  </div>
  {banner}{err_html}
  <div class="fc-pipe">{pipe}</div>
  <div class="fc-body">
    <div class="fc-left">
      <div class="fc-h">🥚 Fridge ({len(chef.fridge_contents)})</div>
      <div style="margin-bottom:13px">{
          _chips(chef.fridge_contents, "g", emoji=True)
          if chef.fridge_contents else '<span class="fc-empty">not checked yet</span>'}</div>
      <div class="fc-h">🛒 Shopping list ({len(chef.shopping_list)})</div>
      <div>{shop_html}</div>
    </div>
    <div class="fc-right">
      <div class="fc-meter"><span>step {step}/{max_steps}</span>
        <span class="fc-track"><span class="fc-fill" style="width:{pct}%"></span></span></div>
      <div class="fc-panel"><div class="fc-h">Recipe</div>{recipe}</div>
      {act_html}
    </div>
  </div>
  <div class="fc-log"><div class="fc-h">Action history</div>{rows}</div>""")


class Live:
    """Redraws one output in place. Falls back to clear_output if the
    update-display protocol misbehaves in your environment."""
    USE_UPDATE_DISPLAY = True

    def __init__(self, display_id="fc-dash"):
        self.id, self.started = display_id, False

    def show(self, html_text: str):
        if not self.USE_UPDATE_DISPLAY:
            clear_output(wait=True)
            display(HTML(html_text))
            return
        if self.started:
            update_display(HTML(html_text), display_id=self.id)
        else:
            display(HTML(html_text), display_id=self.id)
            self.started = True


print("Dashboard ready")

Dashboard ready


In [4]:
#@title Setup 3/3 · Look inside the kitchen (run me) { display-mode: "form" }
_chef0, _world0, _tools0 = create_game()

_TOOL_DOC = [
    ("check_fridge()",                     "list the fridge contents"),
    ("list_recipes()",                     "every recipe in the cookbook"),
    ("search_recipes(ingredient)",         "recipes that use one ingredient"),
    ("get_ingredients(recipe)",            "what a recipe needs, and what you're missing"),
    ("add_to_shopping_list(item)",         "put one missing item on the list"),
    ("done()",                             "stop: recipe chosen, list ready"),
]
_rows = "".join(
    f'<tr><td style="--tc:{TOOL_STYLE[n][1]}"><span style="color:{TOOL_STYLE[n][1]}">'
    f'{TOOL_STYLE[n][0]}</span> <code style="color:{TOOL_STYLE[n][1]};font-weight:700">'
    f'{_esc(sig)}</code></td><td style="color:var(--dim)">{_esc(desc)}</td></tr>'
    for (sig, desc), n in zip(_TOOL_DOC, TOOL_NAMES))

display(HTML(fc_wrap(f"""
<div class="fc-pad">
  <h3>🥚 What's in the fridge</h3>
  <p style="margin-top:0">{_chips(_world0.get_fridge_contents(), "g", emoji=True)}</p>
  <p>The same twelve items for everyone, every single run, so when you compare two agents,
  the only thing that changed is the agent.</p>
  <h3 style="margin-top:18px">🔧 The six tools</h3>
</div>
<table class="fc-tbl"><thead><tr><th>tool</th><th>what it does</th></tr></thead>
<tbody>{_rows}</tbody></table>
<div class="fc-pad"><p style="margin-bottom:0">There are <strong>{len(INGREDIENTS_DB)} recipes</strong>
in the cookbook. The agent can't see any of this directly; the <em>only</em> way it learns
anything about the kitchen is by calling a tool and reading the string that comes back.</p></div>
""")))

tool,what it does
🥚 check_fridge(),list the fridge contents
📚 list_recipes(),every recipe in the cookbook
🔍 search_recipes(ingredient),recipes that use one ingredient
📖 get_ingredients(recipe),"what a recipe needs, and what you're missing"
🛒 add_to_shopping_list(item),put one missing item on the list
✅ done(),"stop: recipe chosen, list ready"


---

## 1 · The rule-based agent

Before using an LLM, let's start with the simplest possible approach: can a few explicit rules solve the task?

We'll build Fridge Chef as a **fixed pipeline**. Every request goes through the same seven steps, in the same order:

1. **Check the fridge.** See which ingredients are available.
2. **Pick an ingredient.** Look for a word in the user's request that matches something in the fridge.
3. **Search recipes.** Find recipes that use that ingredient.
4. **Choose a recipe.** Take the first matching result.
5. **Read its ingredients.** Retrieve everything the recipe requires.
6. **Build the shopping list.** Add any missing ingredients.
7. **Finish.** Return the result.

Most of this pipeline is straightforward to express with ordinary Python code. The fragile part is **step 2**.

For a request such as:

> *"I want to cook something with eggs."*

keyword matching works: the word `eggs` appears in both the request and the fridge.

But consider:

> *"I want something comforting for a cold winter evening."*

There is no ingredient name to match directly. The request expresses a **preference**, not a literal search term.

This is the limitation to watch for in the rule-based version: the pipeline can execute the steps correctly once it knows what to look for, but simple keyword rules struggle to infer that information from less explicit language.

### 🎯 Exercise · Complete the rule-based decision logic

The function `think_rule_based(...)` is called once per turn and must return **exactly one next action**. Your task is to complete **TODO 1a** and **TODO 1b** by following the same pattern: inspect the current state, determine which step has not happened yet, and return the corresponding action.

For **TODO 1a**, once the fridge has been checked, search for recipes using an ingredient extracted from the user's request. For **TODO 1b**, once candidate recipes are available, choose the first one and retrieve its ingredients. Do not change `_extract_ingredient(...)`, the shopping-list logic, or the final `done()` step.

<details>
<summary>💡 Hint for TODO 1a</summary>

Ask yourself: **which field is filled by `search_recipes`?** If that field is still empty, the search has not happened yet.

The fridge is known and we have not searched for recipes yet. The shape is the same as the
`check_fridge` step given just above it; `ingredient` is worked out for you:

```python
if <the field search_recipes fills in is still empty>:
    ingredient = _extract_ingredient(user_request, chef.fridge_contents)
    return <the search_recipes action, with `ingredient` inside it>
```

</details>

<details>
<summary>💡 Hint for TODO 1b</summary>

Ask yourself: **which field is filled by `get_ingredients`?** If that field is still empty, the agent has not committed to a recipe yet.

We have candidate recipes but have not committed to one. Same shape again, with the first
candidate (always the first match: remember that):

```python
if <the field get_ingredients fills in is still empty>:
    recipe = chef.possible_recipes[0]
    return <the get_ingredients action, with `recipe` inside it>
```

</details>

<details>
<summary>💡 Final hint</summary>

The state itself tells you which step comes next:

`no fridge contents → check_fridge`

`no possible recipes → search_recipes`

`no chosen recipe → get_ingredients`

`missing ingredient not yet on shopping list → add_to_shopping_list`

`nothing left to do → done`

Filled in, the two blocks are:

```python
if not chef.possible_recipes:
    ingredient = _extract_ingredient(user_request, chef.fridge_contents)
    return f'ACTION: search_recipes(ingredient="{ingredient}")'
```

```python
if not chef.chosen_recipe:
    recipe = chef.possible_recipes[0]
    return f'ACTION: get_ingredients(recipe="{recipe}")'
```

</details>

In [8]:
def _extract_ingredient(user_request: str, fridge: list[str]) -> str:
    """Find a fridge item mentioned in the request. Given to you: read it, don't change it.

    Works:  "I want to cook something with eggs"          -> "eggs"
    Fails:  "something comforting for a cold evening"      -> falls back to fridge[0]

    This function is the ceiling of the whole rule-based approach. Interpreting what
    somebody *means* is not a string operation.
    """
    text = user_request.lower()
    words = set(re.findall(r"[a-z]+", text))
    for item in fridge:
        if item.lower() in text or set(item.lower().split()) & words:
            return item
    return fridge[0] if fridge else "eggs"          # no idea, just grab the first thing


def think_rule_based(chef: ChefState, world: KitchenWorld,
                     history: list[dict], user_request: str) -> str:
    """Decide the ONE next action. Called once per turn by the loop below.

    Return a string in exactly this shape:
        'ACTION: check_fridge()'
        'ACTION: search_recipes(ingredient="eggs")'
        'ACTION: get_ingredients(recipe="omelette")'
        'ACTION: add_to_shopping_list(item="curry powder")'
        'ACTION: done()'

    Two of those need a value dropped into the string. Use an f-string, and keep the
    inner quotes double so they don't clash with the outer ones:

        ingredient = "eggs"
        return f'ACTION: search_recipes(ingredient="{ingredient}")'
        #      ^ f-string      single quotes outside ^        ^ double quotes inside
        # produces:  ACTION: search_recipes(ingredient="eggs")

    What you can read off `chef` (all empty until the matching tool has run):
        chef.fridge_contents     list[str]
        chef.possible_recipes    list[str]   filled by search_recipes
        chef.chosen_recipe       str         filled by get_ingredients
        chef.needed_ingredients  list[str]   filled by get_ingredients
        chef.shopping_list       list[str]   items added so far
    """

    # ── GIVEN: this is the pattern for the two below ───────────────────────
    # Every step asks the same question: "has this step already happened?" You answer
    # it by looking at the field on `chef` that the step's own tool fills in. Nothing
    # has filled in fridge_contents yet, so that is where the agent starts. Note there
    # is no turn counter anywhere: the state IS the plan.
    if not chef.fridge_contents:
        return 'ACTION: check_fridge()'

    # ── 🎯 TODO 1a ──────────────────────────────────────────────────────────
    if len(chef.possible_recipes) == 0:
      ingredient = _extract_ingredient(user_request, chef.fridge_contents)
      return f"ACTION: search_recipes(ingredient={ingredient})"
    # raise NotImplementedError("🎯 TODO 1a: search for recipes if we haven't yet")

    # ── 🎯 TODO 1b ──────────────────────────────────────────────────────────
    if not chef.chosen_recipe:
      recipe = chef.possible_recipes[0]
      return f"ACTION: get_ingredients(recipe={recipe})"
    #raise NotImplementedError("🎯 TODO 1b: pick the first recipe and read its ingredients")

    # ── given: add missing ingredients one per turn ─────────────────────────
    in_fridge = {i.lower() for i in chef.fridge_contents}
    on_list   = {i.lower() for i in chef.shopping_list}
    for ingredient in chef.needed_ingredients:
        if ingredient.lower() not in in_fridge and ingredient.lower() not in on_list:
            return f'ACTION: add_to_shopping_list(item="{ingredient}")'

    # ── nothing left to buy ─────────────────────────────────────────────────
    return 'ACTION: done()'


print("Rule-based agent defined")

Rule-based agent defined


In [11]:
#@title The rule-based loop (run me) { display-mode: "form" }
# ---------------------------------------------------------------------------
# This is the entire "agent". Notice how little there is to it: ask the think
# function for one action, run it, show the result, repeat. Swapping the think
# function for an LLM later changes nothing about this shape.
# ---------------------------------------------------------------------------
import traceback


def play_rule_based(think_fn, user_request, max_turns=20, delay=0.35):
    chef, world, tools = create_game()
    live, log, history = Live("fc-rules"), [], []
    step, status, error = 0, None, None
    live.show(build_dashboard(user_request, chef, 0, max_turns))

    for _ in range(max_turns):
        # THINK: your code. If it raises, we stop and show you exactly where.
        try:
            action_text = think_fn(chef, world, history, user_request)
            tool_name, args = parse_action(action_text)
        except Exception as exc:
            status = "error"
            error = (f"{type(exc).__name__}: {exc}",
                     "".join(traceback.format_exc().strip().splitlines(keepends=True)[-6:]))
            break

        # ACT
        result = tools.execute(tool_name, args)
        step += 1
        shown = f'{tool_name}({", ".join(f"{k}={v!r}" for k, v in args.items())})'
        history.append({"action": shown, "result": result.message, "success": result.success})
        log.append({"step": step, "action": shown,
                    "result": result.message, "success": result.success})

        live.show(build_dashboard(user_request, chef, step, max_turns,
                                  action=shown, result=result.message,
                                  success=result.success, log=log))

        if chef.is_complete:
            status = "complete"
            break
        # a rule-based loop that never advances would otherwise spin silently
        if len(log) >= 4 and len({e["action"] for e in log[-4:]}) == 1:
            status = "stuck"
            break
        time.sleep(delay)

    status = status or "timeout"
    live.show(build_dashboard(user_request, chef, step, max_turns,
                              log=log, status=status, error=error))
    return {"completed": chef.is_complete, "turns": step,
            "recipe": chef.chosen_recipe or "·",
            "shopping_list": list(chef.shopping_list), "status": status}


print("Loop ready. Now give it something to cook")

Loop ready. Now give it something to cook


### Test 1: a request that names an ingredient

`"I want to cook something with eggs"`. The word *eggs* is right there in the string, so keyword
matching has something to grab. This should work.

In [12]:
REQUEST_CLEAR = "I want to cook something with eggs"

result_rb_clear = play_rule_based(think_rule_based, REQUEST_CLEAR)

### Test 2: a request that names nothing

`"I want something comforting for a cold winter evening"`. Every word a human needs is in there.
Not one of them is a fridge item.

Before you run it: **predict what happens.** Look back at `_extract_ingredient` and work out what
it returns when nothing matches.

In [13]:
REQUEST_VAGUE = "I want something comforting for a cold winter evening"

result_rb_vague = play_rule_based(think_rule_based, REQUEST_VAGUE)

### What just happened?

Both runs completed successfully: no errors, a full shopping list, and a green success banner. Mechanically, the agent did exactly what it was supposed to do, but that does not necessarily mean it made a good decision. Run the cell below and look at **what it actually chose to cook**.

In [14]:
#@title The verdict on rules { display-mode: "form" }
_missing = [n for n in ("result_rb_clear", "result_rb_vague") if n not in globals()]
if _missing:
    print(f"Run the two test cells above first (missing: {', '.join(_missing)}).")
else:
    _same = result_rb_clear["recipe"] == result_rb_vague["recipe"]
    _note = ("<strong>Identical.</strong> Two requests with nothing in common produced the very "
             "same dinner, because the second one fell through to <code>fridge[0]</code> and the "
             "agent searched for eggs. It didn't get the answer wrong so much as it never "
             "considered the question."
             if _same else
             "The recipes differ, but not because anything was <em>understood</em>, only because "
             "the keyword scan happened to land somewhere else.")
    display(HTML(fc_wrap(f"""
<table class="fc-tbl">
  <thead><tr><th>request</th><th>keyword found?</th><th>recipe</th><th class="num">steps</th></tr></thead>
  <tbody>
    <tr><td>{_esc(REQUEST_CLEAR)}</td>
        <td><span class="fc-chip g">✓ "eggs"</span></td>
        <td><strong>{_esc(result_rb_clear['recipe'])}</strong></td>
        <td class="num">{result_rb_clear['turns']}</td></tr>
    <tr><td>{_esc(REQUEST_VAGUE)}</td>
        <td><span class="fc-chip r">✗ nothing matched</span></td>
        <td><strong>{_esc(result_rb_vague['recipe'])}</strong></td>
        <td class="num">{result_rb_vague['turns']}</td></tr>
  </tbody>
</table>
<div class="fc-pad">
  <p style="margin-top:0">{_note}</p>
  <p>This is the honest limit of the approach, and it's worth being precise about it. The agent
  <em>structure</em> was never the problem: the loop, the tools, the state tracking all did their
  job. The one step that needed to map <em>"comforting, cold evening"</em> onto <em>"something warm
  and hearty, so chicken"</em> is the step you cannot write as a rule, because you would have to
  enumerate every way a person might describe being hungry.</p>
  <p style="margin-bottom:0">So keep the loop. Keep the tools. Replace <strong>one function</strong>
  (the part that decides) and see what changes.</p>
</div>""")))

request,keyword found?,recipe,steps
I want to cook something with eggs,"✓ ""eggs""",·,5
I want something comforting for a cold winter evening,✗ nothing matched,·,5


---

## 2 · The LLM agent

We now keep the same kitchen, the same six tools, and the same overall loop, but change one important component: **how the next action is chosen**.

In the rule-based version, that decision was hard-coded in Python. Here, we let an LLM decide which tool to call next based on the user's request and on everything that has happened so far.

The mechanism is fairly simple. Along with the conversation, we send the model a list of available tools, each described in JSON. The model does not execute any Python code itself. Instead, it returns a structured tool call, for example: `search_recipes(ingredient="chicken breast")`. Our code receives that request, runs the corresponding Python function, adds the result back to the conversation, and sends the updated conversation to the model again. This continues until the model decides that the task is complete and calls `done`.

This pattern is usually called **function calling** or **tool calling**. The important idea is that the model is not directly controlling the environment: it is choosing from a set of actions that our code exposes to it.

Two details are especially important. First, the model only knows what the tools return to it, so the information contained in those return values becomes part of its working context. Second, the model decides which tool to use by reading the tool descriptions, so clear and precise descriptions directly affect the quality of the agent's behaviour.

Because this interaction is based on structured tool calls rather than on one specific provider, the same agent loop can later be used with a different model backend. In Section 4, we'll reuse the same structure with a model running locally.

> **You'll need a model from here on.** The next cell sets one up.

In [17]:
#@title Connect a model: pick your backend, then run { display-mode: "form" }
# ---------------------------------------------------------------------------
# Both backends speak the same OpenAI-shaped API, so `client` and `MODEL_NAME`
# are the only two things the rest of the notebook ever sees. Nothing below
# this cell knows or cares which one you chose. Flip it any time and re-run.
# ---------------------------------------------------------------------------
BACKEND = "openrouter"  #@param ["openrouter", "local"]
CLOUD_MODEL = "google/gemini-3.1-flash-lite"  #@param ["google/gemini-3.1-flash-lite", "google/gemini-2.5-flash-lite", "google/gemma-4-31b-it:free"] {allow-input: true}
LOCAL_MODEL = "qwen3:4b-instruct"  #@param ["qwen3:4b-instruct", "granite4.1:3b", "qwen3.5:9b"] {allow-input: true}

import os, shutil, socket, subprocess, urllib.error, urllib.request
from openai import OpenAI

client, MODEL_NAME, BRAIN_LABEL = None, None, None
_PING = [{"role": "user", "content": "Check the fridge."}]
_PING_TOOL = [{"type": "function", "function": {
    "name": "check_fridge", "description": "List what is in the fridge.",
    "parameters": {"type": "object", "properties": {}, "required": []}}}]


def explain_api_error(exc: Exception) -> str:
    """Turn a provider exception into something you can act on."""
    code = getattr(exc, "status_code", None)
    text = str(exc)
    if code == 401 or "No auth credentials" in text:
        return ("401: the API key was rejected. Re-check OPENROUTER_API_KEY in the Secrets "
                "sidebar (🔑 on the left), and that the toggle next to it is ON.")
    if code == 402 or "credit" in text.lower():
        return ("402: this key is out of credit. Ask an instructor, or set BACKEND = 'local' "
                "in the 'Connect a model' cell and run everything from the GPU instead.")
    if code == 429:
        return ("429: rate limited. Free OpenRouter models allow 20 requests/minute and one "
                "agent run costs about 8. Wait a minute, or set BACKEND = 'local'.")
    if code == 404:
        return (f"404: no model called '{MODEL_NAME}'. Model ids get retired; pick another "
                f"from the CLOUD_MODEL dropdown.")
    if isinstance(exc, (ConnectionError, socket.error)) or "Connection" in type(exc).__name__:
        return ("Could not reach the server. On BACKEND='local', run the Ollama setup cell in "
                "section 4 first.")
    return f"{type(exc).__name__}: {text}"


# ── local backend: Ollama on the runtime's GPU ─────────────────────────────
def _sh(cmd: str, timeout: int | None = None) -> str:
    """Run a shell command. On failure, raise with what it actually printed;
    a silent CalledProcessError tells you nothing worth knowing."""
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    if p.returncode != 0:
        detail = ((p.stderr or "") + "\n" + (p.stdout or "")).strip()
        raise RuntimeError(f"`{cmd}` failed (exit {p.returncode}):\n{detail[-1500:]}")
    return p.stdout


def ensure_ollama(model: str, quiet: bool = False) -> None:
    """Install Ollama if needed, start it, and make sure `model` is downloaded."""
    def say(m):
        if not quiet:
            print(m, flush=True)

    gpus = ""
    if shutil.which("nvidia-smi"):
        gpus = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip()
    if not gpus:
        raise RuntimeError(
            "No GPU on this runtime. A 4B model on CPU takes minutes per step.\n"
            "Fix it: Runtime > Change runtime type > Hardware accelerator: T4 GPU > Save.\n"
            "The runtime restarts, so re-run the notebook from the top.")
    say(f"GPU: {gpus.splitlines()[0]}")

    if not shutil.which("ollama"):
        # Ollama 0.34+ ships its Linux build as .tar.zst, and its installer exits
        # with "This version requires zstd for extraction" if zstd is absent,
        # which it is on a stock Colab image. So put that in place first.
        if not shutil.which("zstd"):
            say("Installing zstd (the Ollama installer needs it to unpack)…")
            for attempt in ("apt-get install -y -qq zstd",
                            "apt-get update -qq && apt-get install -y -qq zstd"):
                try:
                    _sh(f"DEBIAN_FRONTEND=noninteractive {attempt} </dev/null", timeout=300)
                    break
                except Exception:
                    continue          # let the installer report the real problem

        say("Installing Ollama (~30 s)…")
        _sh("curl -fsSL https://ollama.com/install.sh | sh", timeout=900)
        if not shutil.which("ollama"):
            raise RuntimeError("The installer reported success but `ollama` is not on PATH.")

    def up() -> bool:
        try:
            urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
            return True
        except Exception:
            return False

    if not up():
        say("Starting the Ollama server…")
        log_path = "/tmp/ollama-serve.log"
        with open(log_path, "wb") as log:      # keep the log; we may need to explain a failure
            subprocess.Popen(["ollama", "serve"], stdout=log, stderr=subprocess.STDOUT)
        for _ in range(60):
            if up():
                break
            time.sleep(1)
        else:
            try:
                tail = open(log_path).read()[-1200:]
            except OSError:
                tail = "(no log written)"
            raise RuntimeError(f"Ollama did not answer within 60 s. Its log said:\n{tail}")

    have = subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout
    if model not in have:
        say(f"Downloading {model}: a few GB, usually under two minutes. Please wait…")
        t0 = time.time()
        pull = subprocess.run(["ollama", "pull", model],
                              capture_output=True, text=True, timeout=1800)
        if pull.returncode != 0:
            raise RuntimeError(
                f"`ollama pull {model}` failed:\n"
                f"{((pull.stderr or '') + (pull.stdout or '')).strip()[-1200:]}\n\n"
                f"If the name is wrong, browse https://ollama.com/library for a valid tag.")
        say(f"Downloaded in {time.time() - t0:.0f} s")


# ── build a client for either backend, on demand ───────────────────────────
def make_client(kind: str):
    """-> (client, model_name, human_label). Used here, and again by the showdown
    in section 4 when it needs both brains in the same cell."""
    if kind == "local":
        ensure_ollama(LOCAL_MODEL)
        return (OpenAI(base_url="http://127.0.0.1:11434/v1", api_key="ollama", timeout=300),
                LOCAL_MODEL, f"{LOCAL_MODEL} · local GPU")

    key = ""
    try:
        from google.colab import userdata
        key = userdata.get("OPENROUTER_API_KEY") or ""
    except Exception:
        key = os.environ.get("OPENROUTER_API_KEY", "")
    if not key:
        display(HTML(fc_wrap("""<div class="fc-pad">
          <h3 style="color:var(--bad)">No API key found</h3>
          <p>Open the <strong>🔑 Secrets</strong> panel on the left of Colab, add a secret named
          <code>OPENROUTER_API_KEY</code>, paste your key, and turn on
          <em>Notebook access</em>. Then re-run this cell.</p>
          <p style="margin-bottom:0">No key handy? Set <code>BACKEND = "local"</code> above and run
          the whole notebook off this runtime's GPU instead, and everything still works.</p>
        </div>""")))
        raise RuntimeError("OPENROUTER_API_KEY is not set (see the panel above).")
    # OPENROUTER_BASE_URL lets a course point everyone at its own gateway or proxy;
    # leave it unset and you talk to OpenRouter directly.
    base = os.environ.get("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")
    return (OpenAI(base_url=base, api_key=key, timeout=120),
            CLOUD_MODEL, CLOUD_MODEL.split("/")[-1])


client, MODEL_NAME, BRAIN_LABEL = make_client(BACKEND)

# ── one cheap round-trip: is it reachable, and can it call tools? ──────────
try:
    _t0 = time.time()
    _probe = client.chat.completions.create(model=MODEL_NAME, messages=_PING,
                                            tools=_PING_TOOL, max_tokens=64)
    _ms = (time.time() - _t0) * 1000
except Exception as _exc:
    raise RuntimeError(explain_api_error(_exc)) from None

_tools_ok = bool(_probe.choices[0].message.tool_calls)
display(HTML(fc_wrap(f"""<div class="fc-pad">
  <h3>Connected</h3>
  <table class="fc-tbl" style="margin-bottom:6px">
    <tr><th>backend</th><td>{_esc(BACKEND)}</td></tr>
    <tr><th>model</th><td><code>{_esc(MODEL_NAME)}</code></td></tr>
    <tr><th>round trip</th><td>{_ms:.0f} ms</td></tr>
    <tr><th>tool calling</th><td>{'<span class="fc-chip g">✓ working</span>' if _tools_ok
        else '<span class="fc-chip r">✗ this model ignored the tool</span>'}</td></tr>
  </table>
  {'' if _tools_ok else '<p style="color:var(--bad)"><strong>Pick a different model.</strong> '
   'This one did not return a tool call, so the agent cannot work. Try another entry in the '
   'dropdown above.</p>'}
</div>""")))

backend,openrouter
model,google/gemini-3.1-flash-lite
round trip,1244 ms
tool calling,✓ working


In [18]:
#@title Tool reference card: what each tool must return { display-mode: "form" }
# The examples below are not written by hand: they are produced by actually running
# the tools, right now. So they cannot drift out of date.
_c, _w, _t = create_game()
_EX = {}
_EX["check_fridge"] = _t.check_fridge().message
_EX["list_recipes"] = _t.list_recipes().message
_EX["search_recipes"] = _t.search_recipes("chicken").message
_EX["get_ingredients"] = _t.get_ingredients("chicken curry").message
_EX["add_to_shopping_list"] = _t.add_to_shopping_list("coconut milk").message
_EX["done"] = _t.done().message

_SPEC = [
    ("check_fridge", "·", "given",
     "Ask the world what's in the fridge and store it on <code>chef.fridge_contents</code>."),
    ("list_recipes", "·", "given",
     "Ask the world for every recipe name, so the agent can browse instead of guessing "
     "which ingredient to search for."),
    ("search_recipes", "ingredient", "yours",
     "Call <code>world.search_recipes(ingredient)</code>. It returns "
     "<code>(matched_name, recipes)</code>, then store the recipe list on "
     "<code>chef.possible_recipes</code>."),
    ("get_ingredients", "recipe", "yours",
     "Call <code>world.get_recipe_ingredients(recipe)</code> → "
     "<code>(matched_name, ingredients)</code>. Store both, on "
     "<code>chef.chosen_recipe</code> and <code>chef.needed_ingredients</code>."),
    ("add_to_shopping_list", "item", "yours",
     "Append one item to <code>chef.shopping_list</code>, but only if it isn't already there "
     "(compare lowercased, or you'll get <em>Rice</em> and <em>rice</em> both on the list)."),
    ("done", "·", "given",
     "Set <code>chef.shopping_done = True</code> and summarise."),
]

_cards = ""
for name, param, who, what in _SPEC:
    emoji, colour, _ = TOOL_STYLE[name]
    tag = ('<span class="fc-chip g">given to you</span>' if who == "given"
           else '<span class="fc-chip a">you write this</span>')
    _cards += f"""
    <div class="fc-act" style="--tc:{colour};margin:10px 16px">
      <div style="display:flex;flex-wrap:wrap;gap:8px;align-items:center">
        <span>{emoji}</span><code>{name}({param if param != '·' else ''})</code>{tag}</div>
      <p style="margin:8px 0 0;font-size:13px;color:var(--dim)">{what}</p>
      <div class="fc-out ok"><strong>returns exactly this shape:</strong>
{_esc(_EX[name])}</div>
    </div>"""

display(HTML(fc_wrap(f"""
<div class="fc-pad" style="padding-bottom:0">
  <h3>The six tools, and what they must give back</h3>
  <p style="margin-top:0">Every tool follows one pattern: <strong>read <code>chef</code> and
  <code>world</code>, do the thing, update <code>chef</code>, return a string.</strong></p>
  <p>That returned string is not for you. It goes straight into the model's context as the
  observation for its next decision. The formatting helpers
  (<code>fmt_search</code>, <code>fmt_ingredients</code>, …) are written for you, so all you have
  to get right is <strong>the state update</strong>.</p>
</div>{_cards}<div style="height:8px"></div>""")))

### 🎯 Exercise · Complete the tool implementations

The LLM can decide which tool to call, but the tools themselves are still ordinary Python functions. In this exercise, you will complete the three missing implementations: `search_recipes`, `get_ingredients`, and `add_to_shopping_list`.

Each function already contains the correct inputs and output formatting. Your job is only to update the agent state correctly before returning the formatted result.

For **TODO 2a**, call `world.search_recipes(ingredient)`, keep both returned values, and store the recipe list in `chef.possible_recipes` when a match is found.

For **TODO 2b**, call `world.get_recipe_ingredients(recipe)` and store both the canonical recipe name and its ingredient list in the chef state.

For **TODO 2c**, append the requested item to `chef.shopping_list`, but only if it is not already present.

<details>
<summary>💡 Hint for TODO 2a</summary>

You need almost exactly this structure:

`matched, recipes = world.search_recipes(ingredient)`

Then, if `recipes` is not empty:

`chef.possible_recipes = recipes`

After that, leave the existing `return fmt_search(...)` unchanged.

</details>

<details>
<summary>💡 Hint for TODO 2b</summary>

Start by calling:

`matched, ingredients = world.get_recipe_ingredients(recipe)`

Then, if `ingredients` is not empty, update both fields:

`chef.chosen_recipe = matched`

`chef.needed_ingredients = ingredients`

Use `matched`, not `recipe`, because `matched` is the canonical recipe name returned by the kitchen.

</details>

<details>
<summary>💡 Hint for TODO 2c</summary>

First check whether the item is already present, ignoring capitalization:

`if item.lower() not in [s.lower() for s in chef.shopping_list]:`

Inside that `if`, append the item:

`chef.shopping_list.append(item)`

Then leave the existing `return fmt_added(...)` unchanged.

</details>

<details>
<summary>💡 Final hint</summary>

Your three missing blocks should follow these patterns:

`matched, recipes = world.search_recipes(ingredient)`

`if recipes:`

`    chef.possible_recipes = recipes`

---

`matched, ingredients = world.get_recipe_ingredients(recipe)`

`if ingredients:`

`    chef.chosen_recipe = matched`

`    chef.needed_ingredients = ingredients`

---

`if item.lower() not in [s.lower() for s in chef.shopping_list]:`

`    chef.shopping_list.append(item)`

</details>

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 🎯 TODO 2: write the three missing tools.
#
# These are the actual Python functions the agent loop will call when the model
# asks for them. Three are done as examples; three are yours. In each one you
# only need the state update, since the message formatting is already handled.
# ═══════════════════════════════════════════════════════════════════════════

_game_state = {"chef": None, "world": None}      # set at the start of each run


# ── given ──────────────────────────────────────────────────────────────────
def check_fridge() -> str:
    chef, world = _game_state["chef"], _game_state["world"]
    chef.fridge_contents = world.get_fridge_contents()
    return fmt_fridge(chef.fridge_contents)


def list_recipes() -> str:
    """Given. Without this the agent can only find a recipe by guessing an
    ingredient that might appear in it, which is exactly how agents fail."""
    chef, world = _game_state["chef"], _game_state["world"]
    chef.possible_recipes = world.all_recipes()
    return fmt_recipes(chef.possible_recipes)


# ── yours ──────────────────────────────────────────────────────────────────
def search_recipes(ingredient: str) -> str:
    chef, world = _game_state["chef"], _game_state["world"]

    raise NotImplementedError("🎯 TODO 2a: search the world, store chef.possible_recipes "
                              "(delete this line once you have)")

    return fmt_search(ingredient, matched, recipes, world.get_fridge_contents())


def get_ingredients(recipe: str) -> str:
    chef, world = _game_state["chef"], _game_state["world"]

    raise NotImplementedError("🎯 TODO 2b: store chef.chosen_recipe and "
                              "chef.needed_ingredients (delete this line once you have)")

    return fmt_ingredients(recipe, matched, ingredients, chef.fridge_contents)


def add_to_shopping_list(item: str) -> str:
    chef = _game_state["chef"]

    raise NotImplementedError("🎯 TODO 2c: append item to chef.shopping_list without "
                              "duplicates (delete this line once you have)")

    return fmt_added(item, chef.shopping_list)


# ── given ──────────────────────────────────────────────────────────────────
def done(unavailable: str = "") -> str:
    """Given. `unavailable` is anything the user asked for that this kitchen
    cannot cook. The model only has to NAME it, and this code puts it on the list."""
    chef = _game_state["chef"]
    if not chef.chosen_recipe:      # refuse to "finish" a job that never started
        return "No recipe chosen yet. Call get_ingredients(recipe=...) first."
    extras = [i.strip() for i in unavailable.split(",") if i.strip()]
    on_list = {s.lower() for s in chef.shopping_list}
    chef.shopping_list.extend(i for i in extras if i.lower() not in on_list)
    chef.unavailable = extras
    chef.shopping_done = True
    return fmt_done(chef.chosen_recipe, chef.shopping_list, extras)


# The loop looks tools up by the name the model gives us.
TOOL_FUNCTIONS = {
    "check_fridge": check_fridge,
    "list_recipes": list_recipes,
    "search_recipes": search_recipes,
    "get_ingredients": get_ingredients,
    "add_to_shopping_list": add_to_shopping_list,
    "done": done,
}
assert set(TOOL_FUNCTIONS) == set(TOOL_NAMES), "tool names are out of sync"
print(f"{len(TOOL_FUNCTIONS)} tools registered: {', '.join(TOOL_FUNCTIONS)}")

In [ ]:
#@title Before TODO 3: what actually reaches the model { display-mode: "form" }
display(HTML(fc_wrap("""
<div class="fc-pad">
  <h3>🧠 The system prompt</h3>
  <p style="margin-top:0">Instructions the model reads <em>before</em> it ever sees the user's
  message: who it is, what the workflow is, how chatty to be. A job description handed to someone
  on day one.</p>
  <p>Ours is part-written. Fill in each <code>???</code> so the workflow is complete and
  unambiguous.</p>

  <div class="fc-act" style="--tc:var(--t-search);margin:12px 0">
    <strong>What a good one actually does</strong>
    <p style="margin:8px 0 0;font-size:13px;color:var(--dim)">Not style advice. These are the
    six things that decide whether the agent works:</p>
    <ol style="font-size:13px;color:var(--dim);margin:8px 0 0;padding-left:20px;line-height:1.7">
      <li><strong>Name the tools exactly.</strong> Write <code>check_fridge</code>, not "look in
      the fridge". The model matches your words against the tool list.</li>
      <li><strong>Number the steps in the order they must happen.</strong> A paragraph gets
      reordered; a numbered list mostly doesn't.</li>
      <li><strong>One <em>action</em> per step.</strong> Guardrails can ride along
      ("…but never search them one at a time"), but never bury a second tool call inside a
      step, and small models do the first and forget the second.</li>
      <li><strong>Say what to do when it can't comply.</strong> No match, nothing suitable,
      impossible request. Left unsaid, the model invents something.</li>
      <li><strong>Say when to stop</strong>, explicitly, naming <code>done</code>.</li>
      <li><strong>Rule out the failure you saw.</strong> Every sentence in a real prompt is
      scar tissue from something going wrong once.</li>
    </ol>
    <p style="margin:10px 0 0;font-size:13px;color:var(--dim)">Then <strong>test it</strong>:
    run the agent, read the <em>Action history</em>, and change one sentence at a time. If you
    can't say which sentence caused a change, you're guessing, not engineering.</p>
  </div>

  <h3 style="margin-top:22px">🔧 The tool descriptions</h3>
  <p style="margin-top:0">Here is the part people find surprising, so read it twice.</p>
  <p>Your Python <strong>docstrings are never sent anywhere.</strong> The model cannot see your
  code. The only thing it knows about your tools is the JSON in <code>OPENAI_TOOLS</code>, and the
  name, the <code>description</code> sentence, and the parameter descriptions.</p>
  <p><strong>Those sentences are prompt.</strong> They are how the model picks between six tools
  it has never encountered before. Four are filled in; two say <code>???</code>, and until you
  write them the model is choosing partly blind.</p>
</div>
<table class="fc-tbl">
  <thead><tr><th>tool</th><th>description the model receives</th></tr></thead>
  <tbody>
    <tr><td>🥚 <code style="color:var(--t-check)">check_fridge</code></td>
        <td><span class="fc-chip a">✏️ you write this</span></td></tr>
    <tr><td>📚 <code style="color:var(--t-list)">list_recipes</code></td>
        <td style="color:var(--dim)">"List every recipe in the cookbook."</td></tr>
    <tr><td>🔍 <code style="color:var(--t-search)">search_recipes</code></td>
        <td style="color:var(--dim)">"Search for recipes that use a specific ingredient."</td></tr>
    <tr><td>📖 <code style="color:var(--t-ingr)">get_ingredients</code></td>
        <td><span class="fc-chip a">✏️ you write this</span></td></tr>
    <tr><td>🛒 <code style="color:var(--t-shop)">add_to_shopping_list</code></td>
        <td style="color:var(--dim)">"Add a missing ingredient to the shopping list."</td></tr>
    <tr><td>✅ <code style="color:var(--t-done)">done</code></td>
        <td style="color:var(--dim)">"Signal that the task is complete…"</td></tr>
  </tbody>
</table>
<div class="fc-pad">
  <p style="margin-bottom:0"><strong>Rule of thumb:</strong> one sentence, active voice, and say
  <em>when</em> to reach for it, not just what it does. Compare
  <em>"Gets ingredients"</em> with <em>"Look up everything a recipe needs, and see what's missing
  from the fridge. Call this once you've chosen a recipe."</em> The second one prevents a whole
  class of wrong turns.</p>
</div>""")))

### 🎯 Exercise · Complete the prompt and two tool descriptions

The system prompt already describes almost the entire workflow the agent should follow. Your task is to complete **step 1** and **step 4**.

**Step 1** is where everything starts. Before the agent can choose anything, it has to find out what the kitchen actually contains.

**Step 4** comes later. At that point the agent has already inspected the fridge, seen the available recipes, and chosen the recipe that best matches the user's request. It now needs to find out **what ingredients that recipe requires** before it can decide what is missing.

Replace each `???` in the prompt with one short instruction that tells the model which tool to call at that point.

The tool list below has the same gap: the descriptions of `check_fridge` and `get_ingredients` are still `???`. The model decides which tool to call by reading those descriptions, so each one needs a sentence that says **what the tool does** and **when to call it**. The other four descriptions show the style.

<details>
<summary>💡 Hint for step 1</summary>

Nothing has happened yet, so the agent knows nothing about the kitchen.

Which tool tells it what is in the fridge?

Your instruction should be very close to:

`Call check_fridge to see what is available.`

</details>

<details>
<summary>💡 Stronger hint for step 1</summary>

The first step is a single tool call, with no arguments:

`check_fridge()`

Tell the model to call it first, before deciding anything else.

</details>

<details>
<summary>💡 Hint for step 4</summary>

The agent already knows the recipe name. Now it needs the recipe's ingredient list.

Which tool is responsible for that?

Your instruction should be very close to:

`Call get_ingredients on the recipe you chose.`

</details>

<details>
<summary>💡 Stronger hint for step 4</summary>

The missing step is a single tool call:

`get_ingredients(recipe=...)`

You do not need to explain the whole workflow again. Just tell the model to call that tool on the recipe it selected in the previous step.

</details>

<details>
<summary>💡 Hint for the descriptions</summary>

A good description answers two questions: *what does this tool do?* and *when should the model call it?*

`check_fridge` lists what the fridge contains, and nothing sensible can happen before it. `get_ingredients` looks up what a recipe needs and which of those items are missing, and it only makes sense once a recipe has been chosen.

</details>

<details>
<summary>💡 Stronger hint for the descriptions</summary>

Two sentences that work:

`List everything currently in the fridge. Call this first, before deciding anything else.`

`Look up everything a recipe needs and report which items are missing from the fridge. Call this once you have chosen a recipe.`

</details>

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 🎯 TODO 3: complete the prompt and two tool descriptions.
#
# Replace each 🎯 ??? : steps 1 and 4 in SYSTEM_INSTRUCTION, and the descriptions
# of check_fridge and get_ingredients in OPENAI_TOOLS.
# ═══════════════════════════════════════════════════════════════════════════

SYSTEM_INSTRUCTION = """You are a kitchen assistant. You plan exactly one dinner, then stop.

Follow this workflow in order:
1. 🎯 ???
2. Call list_recipes to see everything the cookbook can make.
3. Decide what kind of dish the user is asking for (hearty, light, quick, sweet, etc.)
   and choose the recipe that best matches the request.
4. 🎯 ???
5. Call add_to_shopping_list once for each ingredient the recipe is missing.
   Never add something that is already in the fridge.
6. Call done when the task is complete. Put anything the user asked for that this
   kitchen cannot cook in the `unavailable` argument.

A request can be partly impossible. If you can satisfy one part of the request but not another,
prepare the part you can satisfy and report the impossible part through `unavailable`.

Nothing you write down changes the kitchen state. Only tool calls do.
Only use recipe names returned by list_recipes or search_recipes.
Make one tool call at a time. Be brief."""

# The tool list, in the OpenAI function-calling format. THIS is what the model sees:
# not your Python, not your docstrings. Just this JSON.
OPENAI_TOOLS = [
    {"type": "function", "function": {
        "name": "check_fridge",
        "description": "🎯 ???",
        "parameters": {"type": "object", "properties": {}, "required": []},
    }},
    {"type": "function", "function": {
        "name": "list_recipes",
        "description": "List every recipe in the cookbook. Call this early, since it is the "
                       "fastest way to see what can actually be made.",
        "parameters": {"type": "object", "properties": {}, "required": []},
    }},
    {"type": "function", "function": {
        "name": "search_recipes",
        "description": "Search for recipes that use a specific ingredient.",
        "parameters": {"type": "object", "properties": {
            "ingredient": {"type": "string",
                           "description": "An ingredient from the fridge, e.g. 'chicken breast'."},
        }, "required": ["ingredient"]},
    }},
    {"type": "function", "function": {
        "name": "get_ingredients",
        "description": "🎯 ???",
        "parameters": {"type": "object", "properties": {
            "recipe": {"type": "string",
                       "description": "A recipe name you have actually seen, e.g. 'chicken curry'."},
        }, "required": ["recipe"]},
    }},
    {"type": "function", "function": {
        "name": "add_to_shopping_list",
        "description": "Add a missing ingredient to the shopping list.",
        "parameters": {"type": "object", "properties": {
            "item": {"type": "string", "description": "One missing ingredient, e.g. 'curry powder'."},
        }, "required": ["item"]},
    }},
    {"type": "function", "function": {
        "name": "done",
        "description": "Signal that the task is complete. Call this once every missing "
                       "ingredient is on the shopping list.",
        "parameters": {"type": "object", "properties": {
            "unavailable": {"type": "string",
                            "description": "Comma-separated list of anything the user asked "
                                           "for that this kitchen cannot cook at all, e.g. "
                                           "'ice cream'. Leave empty if nothing was missed."},
        }, "required": ["unavailable"]},
    }},
]

In [ ]:
#@title Self-check: is everything described? (run me) { display-mode: "form" }
# A quick self-check, so you find out now rather than three cells later.
# Look for the "???" placeholder specifically, since a single "?" is legitimate prose.
_blank = [t["function"]["name"] for t in OPENAI_TOOLS
          if "???" in t["function"]["description"] or len(t["function"]["description"]) < 15]
_prompt_todo = "???" in SYSTEM_INSTRUCTION
if _prompt_todo:
    print("⚠️  SYSTEM_INSTRUCTION still contains '???', so the agent will improvise those steps.")
if _blank:
    print(f"⚠️  Still to describe: {', '.join(_blank)}")
if not _blank and not _prompt_todo:
    print(f"Prompt is {len(SYSTEM_INSTRUCTION)} characters; "
          f"{len(OPENAI_TOOLS)} tools fully described. Ready.")

In [ ]:
#@title The LLM agent loop (run me) { display-mode: "form" }
# ---------------------------------------------------------------------------
# Compare this to play_rule_based above. Same shape: think, act, observe,
# repeat. The only difference is that "think" is now an HTTP request, and the
# model answers with structured JSON instead of us parsing a string.
# ---------------------------------------------------------------------------
import itertools

_run_seq = itertools.count(1)



def run_agent(user_query: str, max_steps: int = 20, delay: float = 0.3,
              show: bool = True, tools_json=None) -> dict:
    chef, world, _ = create_game()
    _game_state["chef"], _game_state["world"] = chef, world
    tools_json = tools_json if tools_json is not None else OPENAI_TOOLS

    messages = [{"role": "system", "content": SYSTEM_INSTRUCTION},
                {"role": "user", "content": user_query}]
    live = Live(f"fc-agent-{next(_run_seq)}") if show else None
    log, notes = [], []
    step, status, error, nudges = 0, None, None, 0
    tok_in = tok_out = 0
    t0 = time.time()

    if live:
        live.show(build_dashboard(user_query, chef, 0, max_steps, brain=BRAIN_LABEL))

    def ask(force_a_tool: bool):
        """One turn. `force_a_tool` asks the provider to *require* a tool call,
        a real escape hatch for a model that keeps answering in prose. Not every
        provider supports it, so fall back quietly if it is rejected."""
        kw = dict(model=MODEL_NAME, messages=messages, tools=tools_json)
        if force_a_tool:
            try:
                return client.chat.completions.create(**kw, tool_choice="required")
            except Exception:
                pass
        return client.chat.completions.create(**kw)

    while step < max_steps:
        # ── THINK: ask the model which tool to call next ──────────────────
        try:
            resp = ask(force_a_tool=nudges >= 2)
        except Exception as exc:
            status, error = "error", ("The model call failed", explain_api_error(exc))
            break

        usage = getattr(resp, "usage", None)
        if usage:
            tok_in += usage.prompt_tokens or 0
            tok_out += usage.completion_tokens or 0
        msg = resp.choices[0].message
        if msg.content and msg.content.strip():
            notes.append(msg.content.strip())

        # ── the model wrote prose instead of calling a tool ───────────────
        # Small models narrate. Nudge up to three times before giving up, reset the budget
        # after any successful turn, since one stumble at step 2 should not end a run
        # that is going fine at step 12.
        if not msg.tool_calls:
            if nudges >= 3:
                status = "notools"
                break
            nudges += 1
            messages.append({"role": "assistant", "content": msg.content or ""})
            messages.append({"role": "user",
                             "content": "That was text, and text is not recorded. The task "
                                        "is still unfinished. Whatever you just described, "
                                        "do it by calling the tool. If the shopping list is "
                                        "complete, call done now."})
            continue

        nudges = 0
        messages.append({
            "role": "assistant", "content": msg.content or None,
            "tool_calls": [{"id": tc.id, "type": "function",
                            "function": {"name": tc.function.name,
                                         "arguments": tc.function.arguments}}
                           for tc in msg.tool_calls]})

        # ── ACT: run the real Python function the model asked for ─────────
        for tc in msg.tool_calls:
            step += 1
            name = tc.function.name
            try:
                args = json.loads(tc.function.arguments) if tc.function.arguments else {}
            except json.JSONDecodeError:
                args = {}

            fn = TOOL_FUNCTIONS.get(name)
            if fn is None:
                out = f"No such tool '{name}'. Available: {', '.join(TOOL_FUNCTIONS)}."
            else:
                try:
                    out = fn(**args)
                except NotImplementedError as exc:
                    status = "error"
                    error = ("A tool is still unfinished",
                             f"{exc}\n\nGo back to TODO 2 and complete it, then re-run "
                             f"that cell and this one.")
                    break
                except TypeError as exc:
                    out = f"Wrong arguments for {name}: {exc}"

            shown = f'{name}({", ".join(f"{k}={v!r}" for k, v in args.items())})'
            log.append({"step": step, "action": shown, "result": out, "success": True})
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": str(out)})

            if live:
                live.show(build_dashboard(user_query, chef, step, max_steps,
                                          brain=BRAIN_LABEL, action=shown, result=out,
                                          success=True, log=log))
                time.sleep(delay)

        if status == "error" or chef.is_complete:
            break

    status = status or ("complete" if chef.is_complete else "timeout")
    seconds = time.time() - t0

    if live:
        live.show(build_dashboard(user_query, chef, step, max_steps, brain=BRAIN_LABEL,
                                  log=log, status=status, error=error))
        for note in notes:
            display(HTML(fc_wrap(
                f'<div class="fc-pad" style="padding:12px 16px">🤖 <strong>The model also '
                f'said:</strong> <span style="color:var(--dim)">{_esc(note)}</span></div>')))

    return {"completed": chef.is_complete, "status": status, "recipe": chef.chosen_recipe or "·",
            "shopping_list": list(chef.shopping_list), "steps": step, "turns": step,
            "seconds": seconds, "tokens_in": tok_in, "tokens_out": tok_out,
            "model": MODEL_NAME, "backend": BACKEND, "query": user_query}


print(f"Agent ready · brain: {BRAIN_LABEL}")

### The same vague request, now with a model

`"I want something comforting for a cold winter evening"`, the one the rule-based agent answered
with an omelette.

Watch the **first two steps** in particular. The model checks the fridge, and then has to make a
judgement no `if` statement can make: turning *comforting* and *cold evening* into an actual
ingredient it can search for.

In [ ]:
result_llm_vague = run_agent(REQUEST_VAGUE)

---

## 3 · Side by side

Same fridge, same tools, same loop, same request. One function different.

In [ ]:
#@title Rules vs. model, on the same request { display-mode: "form" }
_need = [n for n in ("result_rb_vague", "result_llm_vague") if n not in globals()]
if _need:
    print("Run the rule-based test (section 1) and the LLM run (section 2) first. "
          f"missing: {', '.join(_need)}.")
else:
    def _row(label, a, b, hl=False):
        w = ' style="font-weight:700;color:var(--accent)"' if hl else ""
        return (f"<tr><th>{label}</th><td>{a}</td><td{w}>{b}</td></tr>")

    _rows = "".join([
        _row("recipe chosen", _esc(result_rb_vague["recipe"]),
             _esc(result_llm_vague["recipe"]), hl=True),
        _row("shopping list",
             _esc(", ".join(result_rb_vague["shopping_list"]) or "nothing"),
             _esc(", ".join(result_llm_vague["shopping_list"]) or "nothing")),
        _row("tool calls", result_rb_vague["turns"], result_llm_vague["steps"]),
        _row("how it decided", "keyword scan, then <code>fridge[0]</code>",
             "read the request and inferred an ingredient"),
        _row("cost per run", "free · instant",
             f'{result_llm_vague["tokens_in"] + result_llm_vague["tokens_out"]:,} tokens · '
             f'{result_llm_vague["seconds"]:.1f} s'),
        _row("breaks when…", "the request doesn't name an ingredient",
             "the model misreads intent, or the API is down"),
    ])

    display(HTML(fc_wrap(f"""
<div class="fc-pad" style="padding-bottom:4px">
  <h3>“{_esc(REQUEST_VAGUE)}”</h3>
</div>
<table class="fc-tbl">
  <thead><tr><th></th><th>⚙️ rule-based</th><th>🧠 {_esc(BRAIN_LABEL)}</th></tr></thead>
  <tbody>{_rows}</tbody>
</table>
<div class="fc-pad">
  <p style="margin-top:0">Be careful about the conclusion to draw here. The LLM did not make the
  agent <em>work</em>. The rule-based version completed the task too, and did it faster and for
  free. What the LLM added was <strong>one judgement call</strong> at step two, and that judgement
  is the only reason the answer is any good.</p>
  <p style="margin-bottom:0">That's the real design lesson, and it generalises well past this
  notebook: <strong>use a model for the step that needs understanding, and ordinary code for
  everything else.</strong> Wrapping deterministic work in an LLM buys you latency, cost and
  non-determinism in exchange for nothing.</p>
</div>""")))

---

## 4 · Swap the brain

So far, the agent has used a model through an external API. That works well, but it also introduces a few dependencies: you need an API key, you may have usage costs or rate limits, and your agent depends on someone else's service being available.

Now we'll keep the agent exactly the same and change only the model backend.

Colab gives you access to a **T4 GPU with 16 GB of memory**, which is enough to run a small language model directly inside the notebook. With Ollama, that local model can expose the same OpenAI-compatible interface used by the rest of the code, so switching backends only requires changing one variable:

`BACKEND = "local"`

The important point is that **nothing else changes**. The tools stay the same, the prompt stays the same, and the agent loop stays the same. Only the model that decides which tool to call is replaced. This is possible because the agent depends on a common API shape rather than on one specific provider.

### What changes with a smaller local model?

A local 3B–4B model is much smaller than a frontier model, so you should expect weaker tool-calling behaviour. It may invent recipe names that were never returned by the tools, repeat a search unnecessarily, choose the wrong tool, or forget to call `done`.

Watch for those behaviours when you run the same requests again. The goal here is not simply to make the agent work locally, but to see which parts of the workflow become less reliable when the model is smaller.

At the same time, running locally has important advantages: there is no API key, no per-token cost, no external rate limit, and the conversation stays on the current machine.

### Models that fit comfortably

| model | approximate download | notes |
|---|---:|---|
| `qwen3:4b-instruct` | ~2.5 GB | Default choice for this notebook |
| `granite4.1:3b` | ~2.1 GB | Smaller model with a focus on function calling |
| `qwen3.5:9b` | ~5.5 GB | Better suited to a larger GPU such as an L4 or A100 |

> ⚠️ **Enable a GPU before running the next cell:**  
> *Runtime → Change runtime type → T4 GPU*  
> Running these models on CPU would make each agent step much slower.

In [ ]:
#@title Cloud vs. local, same request { display-mode: "form" }
#@markdown First run downloads the model (~2.5 GB, up to two minutes). Later runs are instant.
SHOWDOWN_QUERY = "I want something comforting for a cold winter evening"  #@param {type:"string"}

import contextlib


@contextlib.contextmanager
def using(kind: str):
    """Temporarily point the agent at a different brain, then put it back."""
    global client, MODEL_NAME, BRAIN_LABEL, BACKEND
    saved = (client, MODEL_NAME, BRAIN_LABEL, BACKEND)
    try:
        client, MODEL_NAME, BRAIN_LABEL = make_client(kind)
        BACKEND = kind
        yield
    finally:
        client, MODEL_NAME, BRAIN_LABEL, BACKEND = saved


_results, _failed = {}, {}
for _kind in ("openrouter", "local"):
    try:
        with using(_kind):
            print(f"→ {BRAIN_LABEL}", flush=True)
            _results[_kind] = run_agent(SHOWDOWN_QUERY, show=False)
    except Exception as _exc:
        _failed[_kind] = explain_api_error(_exc)
        print(f"   skipped: {_failed[_kind]}", flush=True)

if not _results:
    print("Neither backend was reachable, so there is nothing to compare.")
else:
    def _cell(k, fn, dash="·"):
        return fn(_results[k]) if k in _results else dash

    def _tokrate(r):
        return f"{r['tokens_out'] / max(r['seconds'], .001):.0f} tok/s out"

    _hdr = "".join(f"<th>{_esc(_results[k]['model'])}</th>" for k in _results)
    _spec = [
        ("recipe",        lambda r: f"<strong>{_esc(r['recipe'])}</strong>"),
        ("finished cleanly", lambda r: '<span class="fc-chip g">✓ called done()</span>'
            if r["completed"] else f'<span class="fc-chip r">✗ {_esc(r["status"])}</span>'),
        ("tool calls",    lambda r: r["steps"]),
        ("wall clock",    lambda r: f"{r['seconds']:.1f} s"),
        ("tokens",        lambda r: f"{r['tokens_in']:,} in / {r['tokens_out']:,} out"),
        ("throughput",    _tokrate),
        ("shopping list", lambda r: _esc(", ".join(r["shopping_list"]) or "nothing")),
    ]
    _body = "".join(
        f"<tr><th>{label}</th>" + "".join(f"<td>{_cell(k, fn)}</td>" for k in _results) + "</tr>"
        for label, fn in _spec)
    _skipped = "".join(
        f'<p style="color:var(--warn);margin:6px 0 0"><strong>{_esc(k)} skipped:</strong> '
        f'{_esc(v)}</p>' for k, v in _failed.items())

    display(HTML(fc_wrap(f"""
<div class="fc-pad" style="padding-bottom:4px"><h3>“{_esc(SHOWDOWN_QUERY)}”</h3></div>
<table class="fc-tbl"><thead><tr><th></th>{_hdr}</tr></thead><tbody>{_body}</tbody></table>
<div class="fc-pad">{_skipped}
  <p style="margin-top:0">Judge it on the <em>recipe</em> and on <em>finished cleanly</em>, not on
  speed. A small model usually gets to a reasonable dish; where it slips is the bookkeeping:
  looking up a recipe the search never returned, adding an item twice, or reaching the step limit
  because it never called <code>done</code>.</p>
  <p style="margin-bottom:0">If the local run went wrong, don't just accept it: that is a
  <strong>prompt problem</strong> as much as a model problem. Go back to TODO 3, make the workflow
  more explicit: number the steps, say "only use recipe names search_recipes returned", say
  "call done when the list is complete", then run this again. Getting real work out of small
  models is mostly this.</p>
</div>""")))

---

## 5 · One demo is not evidence

So far, you have seen the agent work on a small number of examples. That is useful, but it does not tell you whether the agent is **reliable**.

A successful demo answers one question: *can the agent solve this task?* An evaluation asks a more important one: *how often does it solve the task correctly across different inputs?*

To answer that, we need a fixed set of test cases that we can run every time we change the prompt, the tools, or the model. Even a small suite can reveal obvious regressions: a prompt change that fixes one request but breaks another, a model that forgets to call `done`, or an agent that silently chooses an unrelated recipe.

There is another complication: LLM behaviour is not perfectly deterministic. The same request, with the same prompt and the same model, can produce a different sequence of tool calls on a second run. That means a score such as `4/5` on one run and `3/5` on another does not necessarily mean that anything meaningful changed.

With only five test cases, a one-case difference is especially weak evidence. This suite should therefore be treated as a **smoke test**: it is useful for detecting large failures, but not for proving that one prompt is slightly better than another.

In a real evaluation, you would use many more cases, run them multiple times, and compare the distribution of results rather than relying on a single score.

> **Before drawing conclusions, run the evaluation more than once.** If the score changes between runs, that variation is itself something you should notice.

> **Cost note:** the evaluation runs several agents back to back, which can generate many API calls. If you are using a limited API tier, switch to the local backend before running the full suite.

### 🎯 Exercise · Make one change and evaluate it

Choose **one small change** to the agent, such as improving one sentence in the system prompt or one tool description. Run the evaluation suite before and after the change, then compare the results.

Your goal is not to obtain the highest possible score. Instead, decide whether the evidence is strong enough to claim that your change actually improved the agent.

When comparing the two runs, look at both the final score and the individual failures. A change that moves the score from `3/5` to `4/5` may look better, but with such a small evaluation set that difference could easily be random variation.

<details>
<summary>💡 Hint</summary>

Change only **one thing at a time**. If you edit several parts of the prompt together, you will not know which change caused the new behaviour.

A good experiment is:

`baseline prompt → run eval → change one sentence → run eval again`

</details>

<details>
<summary>💡 Stronger hint</summary>

Do not ask only:

`Did the score go up?`

Also ask:

`Which cases changed?`

`Did one failure disappear while another appeared?`

`Would I see the same result if I ran the suite again?`

</details>

<details>
<summary>💡 Final hint</summary>

With five cases, a one-point improvement is not strong evidence.

If you get:

`before: 3/5`

`after: 4/5`

the safest conclusion is not:

`the new prompt is better`

but:

`the new prompt may be better, but this evaluation is too small and noisy to prove it.`

A large, repeatable difference would be much more convincing.
</details>

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 🎯 TODO 4: add one query of your own to the suite, then read the results.
#
# The interesting entries are the ones you expect to FAIL. A suite of five
# requests the agent already handles teaches you nothing.
# ═══════════════════════════════════════════════════════════════════════════

EVAL_QUERIES = [
    "I want something comforting for a cold winter evening",
    "quick breakfast, I've got five minutes",
    "something light and fresh for lunch",
    "I have guests coming and want to impress them",
    # Replace "???" below with your own request. Pick something awkward: an ingredient
    # this kitchen doesn't stock, two requirements at once, or a flat contradiction.
    "🎯 ???",
]

In [ ]:
#@title Run the suite and score it (run me) { display-mode: "form" }
# ---------------------------------------------------------------------------
# Runs every query in EVAL_QUERIES through run_agent, then renders the
# scorecard. Roughly 40 API requests: mind the free-tier limit noted above.
# ---------------------------------------------------------------------------

if any("???" in q for q in EVAL_QUERIES):
    print("⚠️  EVAL_QUERIES still contains '???'. Replace it in the cell above; running anyway.")

def price_per_token(model: str) -> tuple[float, float]:
    """Live (input, output) price from OpenRouter. Local models are free."""
    if BACKEND != "openrouter":
        return 0.0, 0.0
    base = os.environ.get("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")
    try:
        with urllib.request.urlopen(f"{base}/models", timeout=15) as r:
            for m in json.load(r)["data"]:
                if m["id"] == model:
                    p = m.get("pricing", {})
                    return float(p.get("prompt", 0)), float(p.get("completion", 0))
    except Exception:
        pass
    return 0.0, 0.0


def evaluate(queries) -> list[dict]:
    runs = []
    for i, q in enumerate(queries, 1):
        print(f"[{i}/{len(queries)}] {q[:58]}…", flush=True)
        try:
            runs.append(run_agent(q, show=False))
        except Exception as exc:
            runs.append({"query": q, "completed": False, "status": "crashed",
                         "recipe": "·", "shopping_list": [], "steps": 0, "seconds": 0.0,
                         "tokens_in": 0, "tokens_out": 0, "error": explain_api_error(exc)})
    return runs


eval_runs = evaluate(EVAL_QUERIES)

# ── scorecard ──────────────────────────────────────────────────────────────
_pin, _pout = price_per_token(MODEL_NAME)
_total = sum(r["tokens_in"] * _pin + r["tokens_out"] * _pout for r in eval_runs)
_ok = sum(r["completed"] for r in eval_runs)

def _verdict(run) -> str:
    if run["completed"]:
        return '<span class="fc-chip g">✓</span>'
    return f'<span class="fc-chip r">{_esc(run["status"])}</span>'


_rows = "".join(
    "<tr>"
    f'<td>{_esc(r["query"])}</td>'
    f'<td>{_verdict(r)}</td>'
    f'<td>{_esc(r["recipe"])}</td>'
    f'<td class="num">{r["steps"]}</td>'
    f'<td class="num">{r["seconds"]:.1f} s</td>'
    f'<td class="num">{r["tokens_in"] + r["tokens_out"]:,}</td>'
    "</tr>"
    for r in eval_runs)

_cost = (f"${_total:.4f} for all {len(eval_runs)} runs "
         f"(about ${_total / max(len(eval_runs), 1):.4f} each)"
         if _total else "free, the model is running on this machine")

display(HTML(fc_wrap(f"""
<div class="fc-pad" style="padding-bottom:4px">
  <h3>Scorecard · {_esc(MODEL_NAME)}</h3>
  <p style="margin:0"><strong>{_ok}/{len(eval_runs)}</strong> completed &nbsp;·&nbsp; {_cost}</p>
</div>
<table class="fc-tbl">
  <thead><tr><th>request</th><th>done</th><th>recipe</th><th class="num">calls</th>
  <th class="num">time</th><th class="num">tokens</th></tr></thead>
  <tbody>{_rows}</tbody>
</table>
<div class="fc-pad">
  <p style="margin-top:0"><strong>Read it properly.</strong> "Completed" only means the agent
  called <code>done()</code>, so it is a liveness check, not a quality one. A run can finish
  perfectly and still have chosen a ridiculous dish. Go down the <em>recipe</em> column and ask
  whether each one is something you'd actually want to eat.</p>
  <p style="margin-bottom:0">Then go and change something: one sentence in
  <code>SYSTEM_INSTRUCTION</code>, or one tool description, then re-run this cell. Did the number
  go up? That is the entire discipline. Not "the prompt feels better": a number, on a fixed set
  of inputs, before and after.</p>
</div>""")))

---

## 6 · Your turn

Now try your own requests. Change the query below, run the agent, and inspect not only the final answer but also the **Action history** and the current state.

Some useful cases to try:

- `"What can I cook for breakfast?"` — a straightforward request that should work cleanly.
- `"I want something sweet, and ice cream on the side"` — only partly possible. The cookbook contains a suitable sweet recipe, but no ice cream recipe. A good agent should still choose the sweet dish and report ice cream as unavailable.
- `"I'm vegetarian, no meat please"` — tests whether the model respects a constraint that is not directly encoded in the tools.
- `"I want sushi"` — nothing in the cookbook is a good match. Does the model admit that, or invent something?
- `"cook me something with unicorn meat"` — tests how the agent reacts when a search has no useful result.
- `"I want a big dinner and also a light dessert"` — asks for two things even though the agent is designed to plan only one dinner.

### Trust the actions, not the summary

When an agent fails, the problem is often visible in the tool calls before it appears in the final answer.

For example, the model may write:

> *"Ice cream has been added to the shopping list."*

But if the shopping-list state does not contain `ice cream`, then it was never actually added. Writing that an action happened is not the same as performing the action.

This is why the notebook shows both an **Action history** and a **state panel**. The model's final response is only a description of what it believes happened. The tool calls and state are the actual record of what happened.

> **An agent's summary is not evidence. The tool calls are.**

The same issue can appear when the model writes a closing message but never calls `done()`. From the model's point of view, it may feel finished; from the program's point of view, the task is still incomplete.

### Why `unavailable` is part of `done()`

The `done()` tool requires an `unavailable` argument. Anything the model puts there is handled by code when the task finishes.

This is deliberate. If a behaviour is required, it is safer to make it part of a tool call the model must perform than to rely on the model remembering an extra instruction later.

> **If a behaviour is required, encode it in the interface whenever possible.**

A prompt can encourage a behaviour. A required argument makes that behaviour much harder to forget.

### Tool limitation or model limitation?

When the agent makes a bad decision, ask which part failed.

If the correct recipe was never available to the model, the problem may be in the **tools**. If the right information was available but the model still chose badly, the problem is more likely in the **model or the prompt**.

A useful comparison is to run the same request on both backends. If the larger cloud model succeeds and the smaller local model fails, that suggests a capability difference. If both fail in the same way, inspect the prompt and the tools before blaming the model.

The goal of this section is not just to make the agent succeed. It is to learn how to tell **where** it failed and what evidence you should trust when debugging it.

### 🎯 Exercise · Try a request of your own

Change the request in the cell below, run it, and read the result the way this section asks you
to: the **Action history** and the state panel first, the model's final sentence last. Then try
another one. The requests listed above are good starting points, and the interesting ones are
those you expect to go wrong.

In [ ]:
# ── change this and re-run ─────────────────────────────────────────────────
result = run_agent("I want something sweet, and ice cream on the side")

Everything above presented the agent at a higher level. Now let's look at what the model actually receives.

The payload below contains the complete information available to the model when it makes its first decision: the **system prompt**, the **user message**, and the **JSON definitions of the available tools**. The model then returns a structured response indicating what it wants to do next.

There is no hidden kitchen state inside the model and no special memory of this task. It only works with the context we explicitly send in the request.

In [ ]:
#@title The literal request and response { display-mode: "form" }
_c, _w, _ = create_game()
_game_state["chef"], _game_state["world"] = _c, _w

_payload = {
    "model": MODEL_NAME,
    "messages": [{"role": "system", "content": SYSTEM_INSTRUCTION},
                 {"role": "user", "content": REQUEST_VAGUE}],
    "tools": OPENAI_TOOLS,
}

try:
    _r = client.chat.completions.create(**_payload)
    _calls = _r.choices[0].message.tool_calls or []
    _reply = json.dumps([{"name": c.function.name,
                          "arguments": json.loads(c.function.arguments or "{}")}
                         for c in _calls], indent=2) or "[]"
    _extra = (f'<p style="margin:0 18px 14px;color:var(--dim)">…and that is the whole first turn. '
              f'Our code now runs <code>{_esc(_calls[0].function.name)}</code> for real, appends '
              f'the string it returns to <code>messages</code>, and sends the lot back. Repeat '
              f'until <code>done</code>.</p>' if _calls else "")
except Exception as _exc:
    _reply, _extra = explain_api_error(_exc), ""

display(HTML(fc_wrap(f"""
<div class="fc-pad" style="padding-bottom:0">
  <h3>↑ what we send</h3>
  <p style="margin:0">Note what is <em>absent</em>: your Python source, your docstrings, the recipe
  database, the fridge. The model is told the tools exist and nothing about what they'll say.</p>
</div>
<pre class="code">{_esc(json.dumps(_payload, indent=2))}</pre>
<div class="fc-pad" style="padding-bottom:0;padding-top:6px">
  <h3>↓ what comes back</h3>
  <p style="margin:0">Not text. A request to call a function, with arguments.</p>
</div>
<pre class="code">{_esc(_reply)}</pre>{_extra}""")))

---

## What to take away

| | rule-based | LLM agent |
|---|---|---|
| **decision making** | explicit rules | language understanding |
| **vague requests** | brittle | more flexible |
| **new situations** | usually need new rules | can often generalize |
| **cost & speed** | fast and deterministic | slower, variable, token-based |
| **debugging** | inspect the code | inspect the tool trace and prompt |

The main ideas are simple:

1. **An agent is a loop:** choose an action, run it, observe the result, and repeat until the task is complete.
2. **Tools are ordinary functions:** the model only chooses which function to call and with which arguments.
3. **Prompts and tool descriptions matter:** they directly shape the agent's behaviour.
4. **Use the model where understanding is needed:** keep deterministic logic in code whenever possible.
5. **Trust the trace, not the summary:** tool calls and state tell you what actually happened.

The same agent loop can also work with different model backends. Choosing between a cloud model and a local model is mainly a trade-off between **quality, cost, latency, and privacy**.